# 3D-VSK: 3D Variogram Surface Kriging

Reproducible code accompanying:

**Erarslan, K.** "3D Variogram Surface Kriging: A PSD-Consistent Covariance
Framework for Anisotropic Ore Estimation." *Computers & Geosciences* (submitted).

This notebook reproduces the core methodology and results from the manuscript:
comparison of **Classical Discrete**, **Classical Bilinear** (Erarslan, 2001),
and **3D-VSK (LMC-Ellipsoidal)** covariance formulations on two case studies.

| Cell | Content | Dataset |
|---|---|---|
| 1 | Kalgoorlie Au pipeline (variogram, PSD analysis, LOOCV) | Kalgoorlie Northern Zone (modified, hardcoded) |
| 2 | Synthetic Seyitömer data generator | — |
| 3 | Seyitömer pipeline (variogram, PSD analysis, LOOCV, block model) | Synthetic Seyitömer (see note below) |
| 4 | Variogram surface visualisation + angle sensitivity | Kalgoorlie |

**Data note:** The Kalgoorlie dataset is included directly in Cell 1, modified
from the publicly available Riversgold Limited (2022) quarterly report as
described in the manuscript (Section 3.1). The real Seyitömer-Aslanlı dataset
is **not** redistributed here (see `README.md`); Cell 2 generates a
statistically-equivalent **synthetic** replacement so the full pipeline can
still be run end-to-end. See `README.md` for details.


## Cell 1 — Kalgoorlie Au pipeline
Variogram fitting, PSD analysis, and LOOCV benchmark for the three covariance formulations.


In [ ]:
"""
╔══════════════════════════════════════════════════════════════╗
║         3D-VSK: 3D Variogram Surface Kriging Pipeline        ║
║         Kalgoorlie Au Veri Seti — Tekrarlanabilir Kod        ║
╠══════════════════════════════════════════════════════════════╣
║  Referans : Erarslan (2000) — kongre bildirisi (temel)       ║
║  Geliştirme: Smooth 3D yüzey + LMC + gerçek 3D anizotropi  ║
║  Hedef    : Mathematical Geosciences / Computers & Geosci.   ║
╚══════════════════════════════════════════════════════════════╝

Kullanım (Google Colab veya lokal):
    python 3dvsk_pipeline.py

Çıktılar:
    figures/  — tüm görseller (PNG)
    results/  — metrik tabloları (CSV)
    params/   — fitted variogram parametreleri (JSON)
"""

# ── Bağımlılıklar ─────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import curve_fit
from scipy.interpolate import RBFInterpolator, RegularGridInterpolator
from scipy.linalg import solve
import json, os, warnings
warnings.filterwarnings('ignore')

# ── Çıktı klasörleri ─────────────────────────────────────────
# ── Çıktı dizini: Colab için değiştirebilirsiniz ──────────────
OUTPUT_DIR = "figures"   # Colab: "/content/drive/MyDrive/3dvsk/"
RESULTS_DIR = "results"
PARAMS_DIR  = "params"

for d in [OUTPUT_DIR, RESULTS_DIR, PARAMS_DIR]:
    os.makedirs(d, exist_ok=True)

# ═════════════════════════════════════════════════════════════
# BÖLÜM 0 — VERİ
# ═════════════════════════════════════════════════════════════
class KalgoorlieData:
    """
    Kalgoorlie Altın Projesi — Northern Zone
    20 AC sondaj, koordinatlar /100 düzeltmeli
    Kaynak: Riversgold Limited (akademik kullanım için modifiye)
    """
    SURVEY = {
        "NZAC146":{"x":4120.40,"y":8110.20,"z":162.40},
        "NZAC147":{"x":4130.10,"y":8141.40,"z":162.20},
        "NZAC148":{"x":4140.80,"y":8128.80,"z":162.10},
        "NZAC149":{"x":4151.00,"y":8133.10,"z":162.00},
        "NZAC150":{"x":4102.10,"y":8109.80,"z":161.90},
        "NZAC151":{"x":4141.20,"y":8157.10,"z":161.70},
        "NZAC152":{"x":4130.90,"y":8136.10,"z":161.60},
        "NZAC153":{"x":4151.00,"y":8110.60,"z":161.50},
        "NZAC154":{"x":4101.50,"y":8145.80,"z":161.30},
        "NZAC155":{"x":4111.30,"y":8130.90,"z":161.10},
        "NZAC156":{"x":4122.10,"y":8120.40,"z":162.30},
        "NZAC157":{"x":4113.40,"y":8151.20,"z":162.20},
        "NZAC158":{"x":4142.80,"y":8133.30,"z":162.10},
        "NZAC159":{"x":4153.10,"y":8151.60,"z":161.90},
        "NZAC160":{"x":4153.60,"y":8152.10,"z":161.80},
        "NZAC161":{"x":4142.90,"y":8122.70,"z":161.60},
        "NZAC162":{"x":4123.40,"y":8130.30,"z":161.50},
        "NZAC163":{"x":4133.80,"y":8151.80,"z":161.30},
        "NZAC164":{"x":4103.20,"y":8140.10,"z":161.20},
        "NZAC165":{"x":4114.50,"y":8133.40,"z":161.00},
    }
    ASSAY = {
        "NZAC146":[2.010],"NZAC147":[1.250],"NZAC148":[1.100],
        "NZAC149":[0.610,0.880],"NZAC150":[2.090,1.750],
        "NZAC151":[0.630],"NZAC152":[1.190,1.220],
        "NZAC153":[1.650,1.110],"NZAC154":[1.720,1.410],
        "NZAC155":[1.470],"NZAC156":[1.880],
        "NZAC157":[1.470,1.620],"NZAC158":[0.880],
        "NZAC159":[1.110,1.260],"NZAC160":[1.010,1.330],
        "NZAC161":[1.390],"NZAC162":[1.610],
        "NZAC163":[0.450],"NZAC164":[1.630],
        "NZAC165":[1.910],
    }

    def __init__(self):
        self.hole_ids = sorted(self.SURVEY.keys())
        self.n        = len(self.hole_ids)
        self.coords   = np.array(
            [[self.SURVEY[h]["x"],self.SURVEY[h]["y"],self.SURVEY[h]["z"]]
              for h in self.hole_ids])
        self.au_raw   = np.array(
            [np.mean(self.ASSAY[h]) for h in self.hole_ids])
        self.au_log   = np.log(self.au_raw)

    def summary(self):
        print("─"*55)
        print("VERİ ÖZETİ — Kalgoorlie Au")
        print("─"*55)
        print(f"  n sondaj  : {self.n}")
        print(f"  Au g/t    : min={self.au_raw.min():.3f}, "
              f"max={self.au_raw.max():.3f}, "
              f"mean={self.au_raw.mean():.3f}")
        print(f"  ln(Au)    : mean={self.au_log.mean():.3f}, "
              f"std={self.au_log.std():.3f}")
        print(f"  X aralığı : {self.coords[:,0].min():.1f} – "
              f"{self.coords[:,0].max():.1f} m "
              f"(Δ={self.coords[:,0].max()-self.coords[:,0].min():.1f}m)")
        print(f"  Y aralığı : {self.coords[:,1].min():.1f} – "
              f"{self.coords[:,1].max():.1f} m "
              f"(Δ={self.coords[:,1].max()-self.coords[:,1].min():.1f}m)")
        print(f"  Z aralığı : {self.coords[:,2].min():.2f} – "
              f"{self.coords[:,2].max():.2f} m")


# ═════════════════════════════════════════════════════════════
# BÖLÜM 1 — VARİOGRAM MODELLERİ
# ═════════════════════════════════════════════════════════════
class VariogramModel:
    """Temel variogram/kovaryans fonksiyonları."""

    @staticmethod
    def spherical_gamma(h, C0, C, a):
        """Spherical variogram modeli γ(h)."""
        h = np.atleast_1d(np.array(h, dtype=float))
        g = np.where(h<=0, 0.0,
            np.where(h>=a, C0+C,
                     C0+C*(1.5*h/a - 0.5*(h/a)**3)))
        return g if g.size>1 else float(g)

    @staticmethod
    def spherical_cov(h, C0, C, a):
        """Spherical kovaryans fonksiyonu C(h) = sill - γ(h)."""
        sill = C0 + C
        if np.isscalar(h):
            if h <= 0:  return sill
            if h >= a:  return 0.0
            return sill - (C0 + C*(1.5*h/a - 0.5*(h/a)**3))
        h = np.array(h, dtype=float)
        return np.where(h<=0, sill,
               np.where(h>=a, 0.0,
                        sill-(C0+C*(1.5*h/a-0.5*(h/a)**3))))


# ═════════════════════════════════════════════════════════════
# BÖLÜM 2 — EMPİRİK VARİOGRAM VE MODEL FİTTİNG
# ═════════════════════════════════════════════════════════════
class EmpiricalVariogram:
    """
    Yönlü empirik variogram hesabı ve spherical model fitting.
    """
    DIRECTIONS = [0, 45, 90, 135]
    DIR_LABELS  = ["0° E-W","45° NE-SW","90° N-S","135° NW-SE"]

    def __init__(self, data: KalgoorlieData,
                 angle_tol=22.5, lag_width=8.0,
                 n_lags=7, lag_start=6.0):
        self.data      = data
        self.angle_tol = angle_tol
        self.lag_width = lag_width
        self.n_lags    = n_lags
        self.lag_start = lag_start
        self.empirical = {}   # {dir: {h, gamma, n_pairs}}
        self.fitted    = {}   # {dir: {C0, C, a, R2}}

    def _compute_direction(self, direction_deg):
        """Tek yön için empirik variogram."""
        coords = self.data.coords
        vals   = self.data.au_log
        n      = self.data.n
        dir_r  = np.radians(direction_deg)
        tol_r  = np.radians(self.angle_tol)
        lag_ctrs = self.lag_start + np.arange(self.n_lags)*self.lag_width

        h_out, g_out, np_out = [], [], []
        for lag_h in lag_ctrs:
            lo, hi = lag_h - self.lag_width/2, lag_h + self.lag_width/2
            sq = []
            for i in range(n):
                for j in range(i+1, n):
                    dx = coords[j,0]-coords[i,0]
                    dy = coords[j,1]-coords[i,1]
                    h  = np.sqrt(dx**2+dy**2)
                    if lo <= h < hi and h > 1e-6:
                        pair_ang = np.arctan2(dy, dx)
                        diffs = [abs(pair_ang-dir_r),
                                 abs(pair_ang-dir_r+np.pi),
                                 abs(pair_ang-dir_r-np.pi)]
                        if min(diffs) <= tol_r:
                            sq.append((vals[j]-vals[i])**2)
            if len(sq) >= 2:
                h_out.append(lag_h)
                g_out.append(np.mean(sq)/2.)
                np_out.append(len(sq))

        return (np.array(h_out), np.array(g_out), np.array(np_out))

    def compute_all(self):
        """Tüm yönler için empirik variogram hesapla."""
        for d in self.DIRECTIONS:
            h, g, np_ = self._compute_direction(d)
            self.empirical[d] = {"h":h, "gamma":g, "n_pairs":np_}
        return self

    def fit_all(self):
        """Her yön için spherical model fit et."""
        sill_init = np.var(self.data.au_log)
        for d in self.DIRECTIONS:
            res = self.empirical[d]
            h_d, g_d = res["h"], res["gamma"]
            if len(h_d) < 3:
                # Yetersiz veri — varsayılan
                self.fitted[d] = {"C0":0.01,"C":sill_init*0.9,
                                   "a":40.0,"R2":0.0}
                continue
            try:
                popt, _ = curve_fit(
                    VariogramModel.spherical_gamma, h_d, g_d,
                    p0=[sill_init*0.1, sill_init*0.9, h_d.max()*0.7],
                    bounds=([0,1e-6,1],[sill_init,sill_init*2,100]),
                    maxfev=5000)
                C0f,Cf,af = popt
                g_pred = VariogramModel.spherical_gamma(h_d,*popt)
                ss_res = np.sum((g_d-g_pred)**2)
                ss_tot = np.sum((g_d-g_d.mean())**2)
                r2 = 1-ss_res/ss_tot if ss_tot>0 else 0.0
                self.fitted[d] = {"C0":C0f,"C":Cf,"a":af,"R2":r2}
            except:
                self.fitted[d] = {"C0":0.01,"C":sill_init*0.9,
                                   "a":40.0,"R2":0.0}
        return self

    def print_summary(self):
        print("─"*60)
        print("EMPİRİK VARİOGRAM — Fitted Parametreler")
        print("─"*60)
        print(f"  {'Yön':<15} {'C0':>8} {'C':>8} {'a(m)':>10} "
              f"{'Sill':>8} {'R²':>8}")
        print("  "+"-"*53)
        for d,lbl in zip(self.DIRECTIONS,self.DIR_LABELS):
            p = self.fitted[d]
            print(f"  {lbl:<15} {p['C0']:>8.5f} {p['C']:>8.5f} "
                  f"{p['a']:>10.3f} {p['C0']+p['C']:>8.5f} {p['R2']:>8.4f}")

    def save_params(self, path=os.path.join(PARAMS_DIR,"variogram_params.json")):
        data = {str(d): self.fitted[d] for d in self.DIRECTIONS}
        with open(path,"w") as f:
            json.dump(data, f, indent=2)
        print(f"  Parametreler kaydedildi: {path}")


# ═════════════════════════════════════════════════════════════
# BÖLÜM 3 — 3D VARIOGRAM YÜZEYİ
# ═════════════════════════════════════════════════════════════
class VariogramSurface3D:
    """
    3D Variogram Surface — ana metodolojik katkı.

    Erarslan (2000): bilinear yüzey — yatay yönler
    3D-VSK         : LMC elipsoidal — azimuth + dip

    Kovaryans C(h, azimuth, dip) → sürekli fonksiyon
    """

    def __init__(self, ev: EmpiricalVariogram):
        self.ev   = ev
        self.dirs = ev.DIRECTIONS
        self._build_lmc_params()

    def _build_lmc_params(self):
        """LMC parametrelerini fitted değerlerden türet."""
        fp = self.ev.fitted
        # Sill normalizasyonu — PSD garantisi için
        sills = [fp[d]["C0"]+fp[d]["C"] for d in self.dirs]
        nugs  = [fp[d]["C0"] for d in self.dirs]
        self.sill_norm = np.mean(sills)
        self.nug_norm  = np.mean(nugs)
        self.b_nug     = max(0.010, self.nug_norm)
        self.b_str     = self.sill_norm - self.b_nug

        # Anizotropi parametreleri (range yöne göre)
        ranges = {d: fp[d]["a"] for d in self.dirs}
        self.a_min   = min(ranges.values())
        self.a_max   = max(ranges.values())
        self.theta_max = max(ranges, key=ranges.get)  # en uzun range yönü
        self.a_vert_f = 0.4   # dikey range faktörü

        print(f"\n  LMC Parametreleri:")
        print(f"    b_nugget  = {self.b_nug:.5f}")
        print(f"    b_struct  = {self.b_str:.5f}")
        print(f"    a_min     = {self.a_min:.3f}m  ({[d for d in self.dirs if ranges[d]==self.a_min][0]}°)")
        print(f"    a_max     = {self.a_max:.3f}m  ({self.theta_max}°)")
        print(f"    a_vert_f  = {self.a_vert_f}  (a_vert = a_horiz × {self.a_vert_f})")

    def a_ellipse(self, azimuth_deg):
        """Eliptik anizotropi: yöne bağlı range."""
        t  = np.radians(azimuth_deg % 180)
        tm = np.radians(self.theta_max)
        return self.a_min + (self.a_max-self.a_min)*np.cos(t-tm)**2

    # ── Kovaryans yöntemleri ───────────────────────────────

    def cov_bilinear(self, xi, xj):
        """
        Bilinear yüzey interpolasyonu — Erarslan (2000) yaklaşımı.
        Yalnızca yatay mesafe ve azimuth kullanır.
        """
        if np.allclose(xi,xj): return self.sill_norm
        fp   = self.ev.fitted
        h    = np.linalg.norm(xi[:2]-xj[:2])
        az   = np.degrees(np.arctan2(xj[1]-xi[1],xj[0]-xi[0])) % 180
        dirs = self.dirs + [180]
        vp_ext = {d: fp.get(d, fp[0]) for d in dirs}
        vp_ext[180] = fp[0]
        lo, hi = dirs[-2], dirs[-1]
        for k in range(len(dirs)-1):
            if dirs[k] <= az <= dirs[k+1]:
                lo, hi = dirs[k], dirs[k+1]; break
        p_lo = vp_ext[lo]; p_hi = vp_ext[hi]
        c_lo = VariogramModel.spherical_cov(h,p_lo["C0"],p_lo["C"],p_lo["a"])
        c_hi = VariogramModel.spherical_cov(h,p_hi["C0"],p_hi["C"],p_hi["a"])
        t = (az-lo)/(hi-lo) if hi!=lo else 0.0
        return (1-t)*c_lo + t*c_hi

    def cov_lmc_ellipsoidal(self, xi, xj):
        """
        3D-VSK ana yöntemi: LMC + elipsoidal anizotropi.
        Azimuth VE dip açısını birlikte ele alır.
        PSD yapısal olarak garantilidir (LMC).
        """
        if np.allclose(xi,xj): return self.b_nug + self.b_str
        dx = xj[0]-xi[0]; dy = xj[1]-xi[1]; dz = xj[2]-xi[2]
        h_h  = np.sqrt(dx**2+dy**2)
        az   = np.degrees(np.arctan2(dy,dx)) % 180
        a_h  = self.a_ellipse(az)
        a_v  = a_h * self.a_vert_f
        h_eff = np.sqrt((h_h/a_h)**2+(dz/a_v)**2)*a_h if a_h>1e-6 else abs(dz)
        # Spherical kovaryans (off-diagonal: nugget yok)
        if h_eff <= 0:    c = self.b_str
        elif h_eff >= a_h: c = 0.0
        else: c = self.b_str*(1-(1.5*h_eff/a_h-0.5*(h_eff/a_h)**3))
        return c

    def cov_diagonal(self):
        """Diagonal değer (i=i): nugget + sill."""
        return self.b_nug + self.b_str


# ═════════════════════════════════════════════════════════════
# BÖLÜM 4 — ORDINARY KRİGİNG
# ═════════════════════════════════════════════════════════════
class OrdinaryKriging:
    """
    Ordinary Kriging — 3D-VSK kovaryans fonksiyonu ile.
    """

    def __init__(self, surface: VariogramSurface3D):
        self.surf = surface

    def _build_system(self, coords_tr, cov_func):
        """Kriging sistem matrisi [C|1; 1^T|0]."""
        m  = len(coords_tr)
        diag = self.surf.cov_diagonal()
        K  = np.zeros((m+1,m+1))
        for i in range(m):
            K[i,i] = diag
            for j in range(i+1,m):
                v = cov_func(coords_tr[i],coords_tr[j])
                K[i,j] = v; K[j,i] = v
        K[:m,m] = 1.; K[m,:m] = 1.
        return K

    def predict(self, x0, coords_tr, vals_tr, cov_func):
        """
        Tek nokta tahmini.
        Döndürür: (z_hat, sigma2)
        """
        m  = len(coords_tr)
        K  = self._build_system(coords_tr, cov_func)
        k0 = np.array([cov_func(x0,coords_tr[i]) for i in range(m)]+[1.])
        try:
            w = solve(K, k0, assume_a='sym')
        except:
            w = np.linalg.lstsq(K, k0, rcond=None)[0]
        z_hat  = float(np.dot(w[:m], vals_tr))
        sigma2 = float(self.surf.cov_diagonal() - np.dot(w, k0))
        return z_hat, max(0., sigma2)

    def loocv(self, coords, vals, cov_func, label=""):
        """
        Leave-One-Out Cross Validation.
        Döndürür: dict ile tüm metrikler.
        """
        n = len(vals)
        preds = np.zeros(n)
        vars_ = np.zeros(n)
        for i in range(n):
            idx   = [j for j in range(n) if j!=i]
            z, s2 = self.predict(coords[i], coords[idx],
                                  vals[idx], cov_func)
            preds[i] = z; vars_[i] = s2

        res   = vals - preds
        ss_res = np.sum(res**2)
        ss_tot = np.sum((vals-vals.mean())**2)
        r2    = 1 - ss_res/ss_tot
        msdr  = float(np.mean(res**2/(vars_+1e-10)))
        # Back-transform
        z_bt  = np.exp(preds); v_bt = np.exp(vals)
        rmse_bt = np.sqrt(np.mean((v_bt-z_bt)**2))
        r2_bt   = 1-np.sum((v_bt-z_bt)**2)/np.sum((v_bt-v_bt.mean())**2)

        return {
            "label"  : label,
            "pred"   : preds,
            "var"    : vars_,
            "res"    : res,
            "ME"     : float(np.mean(res)),
            "MAE"    : float(np.mean(np.abs(res))),
            "RMSE"   : float(np.sqrt(np.mean(res**2))),
            "R2"     : float(r2),
            "MSDR"   : msdr,
            "RMSE_bt": rmse_bt,
            "R2_bt"  : r2_bt,
        }


# ═════════════════════════════════════════════════════════════
# BÖLÜM 5 — PSD ANALİZİ
# ═════════════════════════════════════════════════════════════
class PSDAnalysis:
    """Kovaryans matrisinin PSD kontrolü."""

    @staticmethod
    def build_matrix(coords, cov_offdiag, diag_val):
        n = len(coords)
        C = np.zeros((n,n))
        for i in range(n):
            C[i,i] = diag_val
            for j in range(i+1,n):
                v = cov_offdiag(coords[i],coords[j])
                C[i,j]=v; C[j,i]=v
        return C

    @staticmethod
    def check(C, label=""):
        eigs = np.linalg.eigvalsh(C)
        cond = np.linalg.cond(C)
        psd  = eigs.min() > -1e-10
        return {"label":label,"eigs":eigs,
                "lmin":eigs.min(),"lmax":eigs.max(),
                "cond":cond,"psd":psd}

    @staticmethod
    def print_result(res):
        sym = "✓" if res["psd"] else "✗"
        print(f"  {res['label']:<25} λ_min={res['lmin']:>9.5f}  "
              f"koşul={res['cond']:>8.2f}  PSD:{sym}")


# ═════════════════════════════════════════════════════════════
# BÖLÜM 6 — GÖRSELLEŞTİRME
# ═════════════════════════════════════════════════════════════
class Visualizer:
    """Pipeline görselleri."""

    @staticmethod
    def plot_data(data: KalgoorlieData):
        fig,axes=plt.subplots(1,2,figsize=(12,5))
        fig.suptitle("3D-VSK — Veri Özeti: Kalgoorlie Au",
                      fontsize=12,fontweight='bold')
        # Lokasyon haritası
        sc=axes[0].scatter(data.coords[:,0],data.coords[:,1],
                            c=data.au_log,cmap='YlOrRd',
                            s=100,edgecolors='k',linewidths=0.8,zorder=3)
        plt.colorbar(sc,ax=axes[0],label='ln(Au g/t)')
        for i,h in enumerate(data.hole_ids):
            axes[0].annotate(h[-4:],(data.coords[i,0],data.coords[i,1]),
                              fontsize=6,xytext=(0,4),
                              textcoords='offset points',ha='center')
        axes[0].set_xlabel('MGA East (m)'); axes[0].set_ylabel('MGA North (m)')
        axes[0].set_title('Sondaj Lokasyonları — ln(Au g/t)')
        axes[0].grid(True,alpha=0.3)
        # Histogram
        axes[1].hist(data.au_raw,bins=8,color='#e67e22',
                      edgecolor='k',alpha=0.8,label='Au g/t')
        ax2=axes[1].twinx()
        ax2.hist(data.au_log,bins=8,color='#3498db',
                  edgecolor='k',alpha=0.5,label='ln(Au)')
        axes[1].set_xlabel('Au g/t'); axes[1].set_ylabel('Frekans (raw)')
        ax2.set_ylabel('Frekans (log)')
        axes[1].set_title('Au Dağılımı — Raw vs Log')
        lines1,labs1=axes[1].get_legend_handles_labels()
        lines2,labs2=ax2.get_legend_handles_labels()
        axes[1].legend(lines1+lines2,labs1+labs2,fontsize=8)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'fig01_data_summary.png'),dpi=150,bbox_inches='tight')
        plt.close(); print("  → figures/fig01_data_summary.png")

    @staticmethod
    def plot_variograms(ev: EmpiricalVariogram):
        fig,axes=plt.subplots(2,3,figsize=(15,9))
        fig.suptitle("3D-VSK — Empirik Variogram ve Model Fitting",
                      fontsize=12,fontweight='bold')
        axes=axes.flatten()
        cols=['#e74c3c','#e67e22','#27ae60','#2980b9']
        h_fine=np.linspace(0.1,70,300)
        for idx,(d,lbl,col) in enumerate(
                zip(ev.DIRECTIONS,ev.DIR_LABELS,cols)):
            ax=axes[idx]
            res=ev.empirical[d]
            ax.scatter(res["h"],res["gamma"],color=col,
                        s=60,edgecolors='k',linewidths=0.8,
                        zorder=5,label='Empirik')
            for hh,gg,np_ in zip(res["h"],res["gamma"],res["n_pairs"]):
                ax.annotate(f'n={np_}',(hh,gg),xytext=(0,6),
                             textcoords='offset points',fontsize=7,
                             ha='center',color='gray')
            p=ev.fitted[d]
            g_m=VariogramModel.spherical_gamma(h_fine,p["C0"],p["C"],p["a"])
            ax.plot(h_fine,g_m,'-',color=col,lw=2,
                     label=f"Spherical C₀={p['C0']:.3f} C={p['C']:.3f} a={p['a']:.1f}m R²={p['R2']:.3f}")
            ax.axhline(p["C0"]+p["C"],color=col,ls='--',lw=1,alpha=0.5)
            ax.axvline(p["a"],color=col,ls=':',lw=1,alpha=0.5)
            ax.set_xlabel('Lag h (m)'); ax.set_ylabel('γ(h)')
            ax.set_title(f'Variogram — {lbl}')
            ax.legend(fontsize=7); ax.grid(True,alpha=0.3)
            ax.set_xlim(0,72); ax.set_ylim(bottom=0)
        # Tüm yönler birlikte
        ax_all=axes[4]
        for d,lbl,col in zip(ev.DIRECTIONS,ev.DIR_LABELS,cols):
            res=ev.empirical[d]; p=ev.fitted[d]
            ax_all.scatter(res["h"],res["gamma"],color=col,s=35,alpha=0.7)
            g_m=VariogramModel.spherical_gamma(h_fine,p["C0"],p["C"],p["a"])
            ax_all.plot(h_fine,g_m,'-',color=col,lw=2,
                         label=f'{lbl} a={p["a"]:.0f}m')
        ax_all.set_xlabel('Lag h (m)'); ax_all.set_ylabel('γ(h)')
        ax_all.set_title('Tüm Yönler — Anizotropi Özeti')
        ax_all.legend(fontsize=7.5); ax_all.grid(True,alpha=0.3)
        # Anizotropi elipsoidi
        ax_ell=axes[5]
        ranges={d:ev.fitted[d]["a"] for d in ev.DIRECTIONS}
        theta_arr=np.linspace(0,360,360)
        # Basit polar: range yöne göre
        a_min_v=min(ranges.values()); a_max_v=max(ranges.values())
        theta_max_v=max(ranges,key=ranges.get)
        a_arr=[a_min_v+(a_max_v-a_min_v)*
               np.cos(np.radians(t%180)-np.radians(theta_max_v))**2
               for t in theta_arr]
        ax_ell.plot(
            np.array(a_arr)*np.cos(np.radians(theta_arr)),
            np.array(a_arr)*np.sin(np.radians(theta_arr)),
            '-',color='#27ae60',lw=2.5)
        for d,r in ranges.items():
            ax_ell.scatter(r*np.cos(np.radians(d)),
                            r*np.sin(np.radians(d)),
                            s=80,color='#e74c3c',zorder=5)
            ax_ell.annotate(f'{d}° a={r:.0f}m',
                             (r*np.cos(np.radians(d)),
                              r*np.sin(np.radians(d))),
                             fontsize=8,xytext=(4,4),
                             textcoords='offset points')
        ax_ell.set_aspect('equal')
        ax_ell.set_xlabel('Range (m)'); ax_ell.set_ylabel('Range (m)')
        ax_ell.set_title('Anizotropi Elipsoidi')
        ax_ell.grid(True,alpha=0.3)
        ax_ell.axhline(0,color='gray',lw=0.5,alpha=0.4)
        ax_ell.axvline(0,color='gray',lw=0.5,alpha=0.4)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'fig02_variograms.png'),dpi=150,bbox_inches='tight')
        plt.close(); print("  → figures/fig02_variograms.png")

    @staticmethod
    def plot_loocv(data, results_list):
        n_meth=len(results_list)
        fig,axes=plt.subplots(2,n_meth,figsize=(5*n_meth,10))
        fig.suptitle("3D-VSK — LOOCV Karşılaştırması",
                      fontsize=13,fontweight='bold')
        au_log=data.au_log; au_raw=np.exp(au_log)
        lim=[au_log.min()-0.15,au_log.max()+0.15]
        lim_bt=[0,2.3]
        cols_m=['#95a5a6','#e67e22','#27ae60','#8e44ad']
        for idx,r in enumerate(results_list):
            col=cols_m[idx%len(cols_m)]
            # Log uzayı scatter
            ax=axes[0,idx]
            sc=ax.scatter(au_log,r["pred"],
                           c=np.abs(r["res"]),cmap='RdYlGn_r',
                           vmin=0,vmax=0.5,s=65,
                           edgecolors='k',linewidths=0.7,zorder=4)
            ax.plot(lim,lim,'k--',lw=1.5,alpha=0.6)
            plt.colorbar(sc,ax=ax,label='|residual|')
            ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect('equal')
            ax.set_xlabel('Gerçek ln(Au)'); ax.set_ylabel('Tahmin ln(Au)')
            ax.set_title(f"{r['label']}\nRMSE={r['RMSE']:.4f} R²={r['R2']:.4f}",
                          fontsize=9,fontweight='bold')
            ax.text(0.05,0.92,f"MSDR={r['MSDR']:.3f}",
                     transform=ax.transAxes,fontsize=8,
                     bbox=dict(boxstyle='round',fc='white',alpha=0.8))
            ax.grid(True,alpha=0.3)
            # Back-transform scatter
            ax2=axes[1,idx]
            z_bt=np.exp(r["pred"])
            ax2.scatter(au_raw,z_bt,color=col,s=65,
                         edgecolors='k',linewidths=0.7,zorder=4)
            ax2.plot(lim_bt,lim_bt,'k--',lw=1.5,alpha=0.6)
            ax2.set_xlim(lim_bt); ax2.set_ylim(lim_bt); ax2.set_aspect('equal')
            ax2.set_xlabel('Gerçek Au g/t'); ax2.set_ylabel('Tahmin Au g/t')
            ax2.set_title(f"Back-transform\nRMSE={r['RMSE_bt']:.4f} "
                           f"R²={r['R2_bt']:.4f}",fontsize=9)
            ax2.grid(True,alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'fig03_loocv.png'),dpi=150,bbox_inches='tight')
        plt.close(); print("  → figures/fig03_loocv.png")

    @staticmethod
    def plot_benchmark(results_list, psd_results):
        fig,axes=plt.subplots(1,3,figsize=(16,6))
        fig.suptitle("3D-VSK — Benchmark Özeti: Klasik vs 3D-VSK",
                      fontsize=12,fontweight='bold')
        names =[r["label"] for r in results_list]
        rmses =[r["RMSE"]  for r in results_list]
        r2s   =[r["R2"]    for r in results_list]
        lmins =[p["lmin"]  for p in psd_results]
        cols_b=['#95a5a6','#e67e22','#27ae60','#8e44ad']
        x=np.arange(len(names))
        # RMSE
        bars=axes[0].bar(x,rmses,color=cols_b,edgecolor='k',
                          linewidth=0.8,alpha=0.85)
        for bar,v in zip(bars,rmses):
            axes[0].text(bar.get_x()+bar.get_width()/2,
                          v+0.005,f'{v:.4f}',
                          ha='center',va='bottom',fontsize=9,fontweight='bold')
        axes[0].set_xticks(x); axes[0].set_xticklabels(names,fontsize=9)
        axes[0].set_ylabel('RMSE (log uzayı)'); axes[0].set_title('RMSE Karşılaştırması')
        axes[0].grid(True,alpha=0.3,axis='y')
        # R²
        bars2=axes[1].bar(x,r2s,color=cols_b,edgecolor='k',
                           linewidth=0.8,alpha=0.85)
        for bar,v in zip(bars2,r2s):
            ypos=max(v,0)+0.01
            axes[1].text(bar.get_x()+bar.get_width()/2,
                          ypos,f'{v:.4f}',
                          ha='center',va='bottom',fontsize=9,fontweight='bold')
        axes[1].set_xticks(x); axes[1].set_xticklabels(names,fontsize=9)
        axes[1].set_ylabel('R²'); axes[1].set_title('R² Karşılaştırması')
        axes[1].grid(True,alpha=0.3,axis='y')
        # λ_min
        bar_cols_psd=['#27ae60' if v>-1e-10 else '#e74c3c' for v in lmins]
        bars3=axes[2].bar(x,lmins,color=bar_cols_psd,
                           edgecolor='k',linewidth=0.8,alpha=0.85)
        axes[2].axhline(0,color='black',lw=2,ls='--')
        axes[2].fill_between([-0.5,len(names)-0.5],
                              [-0.5,-0.5],[0,0],
                              alpha=0.08,color='red',label='PSD ihlal bölgesi')
        for bar,v in zip(bars3,lmins):
            axes[2].text(bar.get_x()+bar.get_width()/2,
                          v-(0.02 if v<0 else -0.01),
                          f'{v:.4f}',ha='center',
                          va='top' if v<0 else 'bottom',
                          fontsize=9,fontweight='bold',
                          color='#c0392b' if v<-1e-10 else '#1e8449')
        axes[2].set_xticks(x); axes[2].set_xticklabels(names,fontsize=9)
        axes[2].set_ylabel('λ_min'); axes[2].set_title('PSD Analizi (λ_min)')
        axes[2].legend(fontsize=8); axes[2].grid(True,alpha=0.3,axis='y')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'fig04_benchmark.png'),dpi=150,bbox_inches='tight')
        plt.close(); print("  → figures/fig04_benchmark.png")


# ═════════════════════════════════════════════════════════════
# ANA PIPELINE
# ═════════════════════════════════════════════════════════════
def run_pipeline():
    print("\n" + "═"*62)
    print("  3D-VSK PIPELINE — Başlatılıyor")
    print("═"*62)

    # ── Faz 0: Veri ────────────────────────────────────────
    print("\n[FAZ 0] Veri yükleme...")
    data = KalgoorlieData()
    data.summary()
    print("\n  Görsel oluşturuluyor...")
    Visualizer.plot_data(data)

    # ── Faz 1-2: Variogram ─────────────────────────────────
    print("\n[FAZ 1-2] Empirik variogram ve model fitting...")
    ev = EmpiricalVariogram(data)
    ev.compute_all()
    ev.fit_all()
    ev.print_summary()
    ev.save_params()
    print("\n  Görsel oluşturuluyor...")
    Visualizer.plot_variograms(ev)

    # ── Faz 3: 3D Yüzey ────────────────────────────────────
    print("\n[FAZ 3] 3D Variogram Surface inşa ediliyor...")
    surf = VariogramSurface3D(ev)

    # ── Faz 4: PSD Kontrolü ────────────────────────────────
    print("\n[FAZ 4] PSD Analizi...")
    print("─"*55)
    print(f"  {'Yöntem':<25} {'λ_min':>9} {'Koşul':>9} {'PSD':>6}")
    print("  "+"-"*51)

    psd_results = []

    # Klasik: ayrıksal seçim
    def nearest_dir(az):
        base=az%180
        diffs=[min(abs(base-d),180-abs(base-d))
               for d in ev.DIRECTIONS]
        return ev.DIRECTIONS[np.argmin(diffs)]

    def cov_disc(xi,xj):
        if np.allclose(xi,xj): return ev.fitted[0]["C0"]+ev.fitted[0]["C"]
        h=np.linalg.norm(xi[:2]-xj[:2])
        az=np.degrees(np.arctan2(xj[1]-xi[1],xj[0]-xi[0]))%180
        d=nearest_dir(az); p=ev.fitted[d]
        return VariogramModel.spherical_cov(h,p["C0"],p["C"],p["a"])

    for label,cfunc,diag in [
        ("Classical Discrete",  cov_disc,                  ev.fitted[0]["C0"]+ev.fitted[0]["C"]),
        ("Classical Bilinear",  surf.cov_bilinear,          surf.sill_norm),
        ("3D-VSK LMC-Ellip",   surf.cov_lmc_ellipsoidal,   surf.cov_diagonal()),
    ]:
        C_m = PSDAnalysis.build_matrix(data.coords, cfunc, diag)
        res = PSDAnalysis.check(C_m, label)
        PSDAnalysis.print_result(res)
        psd_results.append(res)

    # ── Faz 5: LOOCV ───────────────────────────────────────
    print("\n[FAZ 5] LOOCV Benchmark...")
    print("─"*70)
    print(f"  {'Yöntem':<25} {'ME':>8} {'MAE':>8} {'RMSE':>8} "
          f"{'R²':>8} {'MSDR':>8}")
    print("  "+"-"*65)

    ok = OrdinaryKriging(surf)
    loocv_results = []

    for label,cfunc in [
        ("Classical Discrete",  cov_disc),
        ("Classical Bilinear",  surf.cov_bilinear),
        ("3D-VSK LMC-Ellip",   surf.cov_lmc_ellipsoidal),
    ]:
        r = ok.loocv(data.coords, data.au_log, cfunc, label)
        loocv_results.append(r)
        print(f"  {label:<25} {r['ME']:>+8.5f} {r['MAE']:>8.5f} "
              f"{r['RMSE']:>8.5f} {r['R2']:>8.5f} {r['MSDR']:>8.5f}")

    # ── Görseller ──────────────────────────────────────────
    print("\n[GÖRSEL] Sonuç grafikleri oluşturuluyor...")
    Visualizer.plot_loocv(data, loocv_results)
    Visualizer.plot_benchmark(loocv_results, psd_results)

    # ── Sonuç kaydetme ─────────────────────────────────────
    import csv
    with open(os.path.join(RESULTS_DIR,"loocv_metrics.csv"),"w",newline="") as f:
        w=csv.writer(f)
        w.writerow(["Method","ME","MAE","RMSE","R2","MSDR","RMSE_bt","R2_bt"])
        for r in loocv_results:
            w.writerow([r["label"],r["ME"],r["MAE"],r["RMSE"],
                        r["R2"],r["MSDR"],r["RMSE_bt"],r["R2_bt"]])
    print("  → results/loocv_metrics.csv")

    # ── Final özet ─────────────────────────────────────────
    print("\n" + "═"*62)
    print("  3D-VSK PIPELINE — Tamamlandı")
    print("═"*62)
    print(f"\n  En iyi LOOCV performansı:")
    best = min(loocv_results, key=lambda r: r["RMSE"])
    print(f"    Yöntem : {best['label']}")
    print(f"    RMSE   : {best['RMSE']:.5f}")
    print(f"    R²     : {best['R2']:.5f}")
    print(f"    MSDR   : {best['MSDR']:.5f}")
    print(f"\n  PSD durumu:")
    for p in psd_results:
        sym="✓" if p["psd"] else "✗"
        print(f"    {p['label']:<25} {sym}")
    print()

# ─────────────────────────────────────────────
if __name__ == "__main__":
    run_pipeline()

## Cell 2 — Synthetic Seyitömer data generator
Generates `synthetic_seyitomer.csv`: a statistically-equivalent replacement for the real (non-redistributable) Seyitömer-Aslanlı dataset. See script docstring for full methodology.


In [ ]:
#!/usr/bin/env python3
"""
================================================================================
Synthetic Seyitömer-Aslanlı Dataset Generator
================================================================================
Generates a statistically-equivalent SYNTHETIC replacement for the real
Seyitömer-Aslanlı drill hole dataset used in the 3D-VSK manuscript
(Erarslan, "3D Variogram Surface Kriging: A PSD-Consistent Covariance
Framework for Anisotropic Ore Estimation", Computers & Geosciences).

WHY SYNTHETIC DATA:
The real Seyitömer-Aslanlı drill hole data was obtained during a site visit
to a now-closed lignite mining operation. Although the mine is closed, the
underlying exploration/production data is not the author's to redistribute
publicly. To preserve full reproducibility of the 3D-VSK methodology without
disclosing proprietary survey data, this script generates a synthetic dataset
that:

  1. Has the same sample size (n=191) and spatial extent (3,736 x 6,511 m)
     as the real dataset (Section 4.1 of the manuscript).
  2. Reproduces the same directional variogram structure (nugget, sill,
     range per principal direction) reported in Table 9 of the manuscript,
     via simulation from the fitted LMC-Ellipsoidal covariance model.
  3. Reproduces the same marginal statistics (mean, std, skewness) for
     seam thickness and calorific value reported in Table 8.
  4. Reproduces the same large-scale spatial trend (X/Y correlation)
     reported in Section 4.2 (non-stationarity).

This is NOT the real Seyitömer dataset. Absolute values, drill hole
identities, and exact spatial positions carry no geological meaning.
The synthetic dataset exists solely so that the 3D-VSK / Classical
Discrete / Classical Bilinear comparison code can be run end-to-end by
anyone, producing qualitatively identical PSD-violation and LOOCV
behaviour to that reported in the manuscript.

Usage:
    python synthetic_seyitomer_generator.py
    # writes synthetic_seyitomer.csv to the current directory

Dependencies: numpy, pandas, scipy
================================================================================
"""

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

RANDOM_SEED = 35  # validated against target statistics; see validation block below

# ── Field geometry (Section 4.1) ──────────────────────────────────────────
N_HOLES   = 191
X_EXTENT  = 3736.0   # m
Y_EXTENT  = 6511.0   # m

# ── Target marginal statistics (Table 8) ──────────────────────────────────
THICKNESS_TARGET = dict(mean=13.3, std=8.1, min_clip=0.5, max_clip=43.5)
SQRT_THICKNESS_TARGET = dict(mean=3.47, std=1.10)  # Table 8, sqrt(Thickness) row
CALORIFIC_TARGET = dict(mean=2039.0, std=374.0, min_clip=1088.0, max_clip=3187.0)

# ── Fitted directional LMC variogram parameters (Table 9) ────────────────
# sqrt(Thickness) space
THICKNESS_VARIO = {
    0:   dict(C0=0.654, C=0.890, a=1245.0),
    45:  dict(C0=0.449, C=1.719, a=1245.0),
    90:  dict(C0=0.342, C=0.866, a=1383.0),
    135: dict(C0=0.553, C=0.642, a=1245.0),
}
# Calorific value (kcal/kg) space
CALORIFIC_VARIO = {
    0:   dict(C0=75588.0,  C=106301.0, a=1245.0),
    45:  dict(C0=63436.0,  C=133055.0, a=1245.0),
    90:  dict(C0=78940.0,  C=55530.0,  a=317.0),
    135: dict(C0=103179.0, C=4642.0,   a=1245.0),
}

# ── Target large-scale trend (Section 4.2) ────────────────────────────────
# Pearson r between variable and X, Y coordinates
THICKNESS_TREND = dict(r_x=0.189, r_y=0.344)
CALORIFIC_TREND = dict(r_x=0.465, r_y=0.128)


def ellipsoidal_range(theta_deg, vario_dict):
    """Direction-dependent range a(theta) via cosine-squared interpolation
    between the fitted directional ranges (mirrors Eq. 10 of the manuscript).
    """
    dirs = np.array(sorted(vario_dict.keys()))
    ranges = np.array([vario_dict[d]['a'] for d in dirs])
    a_min, a_max = ranges.min(), ranges.max()
    theta_max = dirs[np.argmax(ranges)]
    t = np.radians(theta_deg % 180)
    tm = np.radians(theta_max)
    return a_min + (a_max - a_min) * np.cos(t - tm) ** 2


def mean_sill(vario_dict):
    return np.mean([v['C0'] + v['C'] for v in vario_dict.values()])


def lmc_covariance_matrix(coords, vario_dict):
    """Builds an LMC-Ellipsoidal covariance matrix (same construction as
    3D-VSK, Eq. 12) for an arbitrary point set, used here purely as the
    generating covariance for the synthetic Gaussian field.
    """
    n = coords.shape[0]
    S_bar = mean_sill(vario_dict)
    K = np.full((n, n), S_bar)
    for i in range(n):
        for j in range(i + 1, n):
            dx = coords[j, 0] - coords[i, 0]
            dy = coords[j, 1] - coords[i, 1]
            h = np.hypot(dx, dy)
            az = np.degrees(np.arctan2(dy, dx)) % 180
            a_theta = ellipsoidal_range(az, vario_dict)
            if h >= a_theta:
                cov = 0.0
            else:
                cov = S_bar * (1 - 1.5 * (h / a_theta) + 0.5 * (h / a_theta) ** 3)
            K[i, j] = K[j, i] = cov
    return K


def simulate_gaussian_field(coords, vario_dict, rng):
    """Simulates a zero-mean, unit-ish-variance spatially correlated
    Gaussian field via Cholesky factorisation of the LMC covariance matrix.
    """
    K = lmc_covariance_matrix(coords, vario_dict)
    # Diagonal jitter for numerical stability. The LMC construction is
    # PSD-consistent in principle (see manuscript Eq. 12), but finite-
    # precision arithmetic on a 191x191 matrix with strongly anisotropic
    # parameters (e.g. the calorific-value case) can still produce tiny
    # negative eigenvalues; a small relative jitter resolves this without
    # affecting the simulated spatial structure.
    eigval_min = np.linalg.eigvalsh(K).min()
    jitter = max(1e-6, abs(min(0.0, eigval_min)) * 1.5) * np.trace(K) / K.shape[0]
    K += np.eye(K.shape[0]) * jitter
    L = np.linalg.cholesky(K)
    z = rng.standard_normal(K.shape[0])
    return L @ z


def shape_marginal(field, target_mean, target_std, min_clip, max_clip, rng,
                    skew_strength=0.0):
    """Rescales a standard-normal-ish field to the target mean/std, applies
    a mild skew transform if requested, and clips to the observed data range.
    """
    z = (field - field.mean()) / field.std()
    if skew_strength != 0.0:
        z = z + skew_strength * (z ** 2 - 1.0)  # adds positive skew
        z = (z - z.mean()) / z.std()
    values = target_mean + target_std * z
    return np.clip(values, min_clip, max_clip)


def add_trend(values, coords, r_x, r_y, rng):
    """Blends in a linear X/Y trend at approximately the target Pearson
    correlation strength, preserving the marginal mean/std as closely as
    possible.
    """
    x_n = (coords[:, 0] - coords[:, 0].mean()) / coords[:, 0].std()
    y_n = (coords[:, 1] - coords[:, 1].mean()) / coords[:, 1].std()
    v_n = (values - values.mean()) / values.std()

    # Simple linear combination targeting approximate correlation magnitude
    trend = r_x * x_n + r_y * y_n
    blended = np.sqrt(max(1e-6, 1 - r_x**2 - r_y**2)) * v_n + trend
    blended = blended * values.std() + values.mean()
    return blended


def generate_synthetic_seyitomer(seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)

    # ── 1. Drill hole locations ──
    coords = rng.uniform([0, 0], [X_EXTENT, Y_EXTENT], size=(N_HOLES, 2))
    tree = cKDTree(coords)
    d, _ = tree.query(coords, k=2)
    print(f"Synthetic mean NN distance: {d[:, 1].mean():.1f} m "
          f"(target from real dataset: 185.5 m)")

    # ── 2. Simulate sqrt(Thickness) field ──
    sqrt_thick_field = simulate_gaussian_field(coords, THICKNESS_VARIO, rng)
    sqrt_thick = shape_marginal(
        sqrt_thick_field,
        target_mean=SQRT_THICKNESS_TARGET['mean'],
        target_std=SQRT_THICKNESS_TARGET['std'],
        min_clip=np.sqrt(THICKNESS_TARGET['min_clip']),
        max_clip=np.sqrt(THICKNESS_TARGET['max_clip']),
        rng=rng, skew_strength=0.05,
    )
    sqrt_thick = add_trend(sqrt_thick, coords, **THICKNESS_TREND, rng=rng)
    thickness = np.clip(sqrt_thick, 0, None) ** 2
    thickness = np.clip(thickness, THICKNESS_TARGET['min_clip'],
                         THICKNESS_TARGET['max_clip'])

    # ── 3. Simulate calorific value field ──
    cal_field = simulate_gaussian_field(coords, CALORIFIC_VARIO, rng)
    calorific = shape_marginal(
        cal_field,
        target_mean=CALORIFIC_TARGET['mean'],
        target_std=CALORIFIC_TARGET['std'],
        min_clip=CALORIFIC_TARGET['min_clip'],
        max_clip=CALORIFIC_TARGET['max_clip'],
        rng=rng, skew_strength=0.03,
    )
    calorific = add_trend(calorific, coords, **CALORIFIC_TREND, rng=rng)
    calorific = np.clip(calorific, CALORIFIC_TARGET['min_clip'],
                         CALORIFIC_TARGET['max_clip'])

    # ── 4. Synthetic seam roof elevation (coal_top), for 3D block model ──
    # Loosely correlated with a broad regional dip, consistent with
    # Section 4.1 (coal_top range 1,040-1,181 m a.s.l.)
    regional_dip = (coords[:, 0] / X_EXTENT) * 80 + (coords[:, 1] / Y_EXTENT) * 60
    coal_top = 1040 + regional_dip + rng.normal(0, 8, N_HOLES)
    coal_top = np.clip(coal_top, 1040, 1181)

    df = pd.DataFrame({
        'hole_id': [f'SYN{i+1:04d}' for i in range(N_HOLES)],
        'x': coords[:, 0],
        'y': coords[:, 1],
        'thickness_m': thickness,
        'calorific_kcal_kg': calorific,
        'coal_top_masl': coal_top,
    })
    return df


if __name__ == '__main__':
    df = generate_synthetic_seyitomer()
    df.to_csv('synthetic_seyitomer.csv', index=False)

    print("\n=== Synthetic dataset summary (compare to manuscript Table 8) ===")
    print(f"n = {len(df)}")
    print(f"Thickness:  min={df.thickness_m.min():.1f}  max={df.thickness_m.max():.1f}  "
          f"mean={df.thickness_m.mean():.1f}  std={df.thickness_m.std():.1f}")
    print(f"Calorific:  min={df.calorific_kcal_kg.min():.0f}  max={df.calorific_kcal_kg.max():.0f}  "
          f"mean={df.calorific_kcal_kg.mean():.0f}  std={df.calorific_kcal_kg.std():.0f}")

    from scipy.stats import pearsonr
    rx_t, _ = pearsonr(df.thickness_m, df.x)
    ry_t, _ = pearsonr(df.thickness_m, df.y)
    rx_c, _ = pearsonr(df.calorific_kcal_kg, df.x)
    ry_c, _ = pearsonr(df.calorific_kcal_kg, df.y)
    print(f"\nThickness trend:  r_X={rx_t:.3f} (target 0.189)  r_Y={ry_t:.3f} (target 0.344)")
    print(f"Calorific trend:  r_X={rx_c:.3f} (target 0.465)  r_Y={ry_c:.3f} (target 0.128)")
    print(f"\nWrote synthetic_seyitomer.csv ({len(df)} rows)")


## Cell 3 — Seyitömer pipeline (synthetic data)
Variogram fitting, PSD analysis, LOOCV, and 3D block model — run on the synthetic dataset generated above.


In [ ]:
"""
╔══════════════════════════════════════════════════════════════╗
║   3D-VSK Pipeline v2 — SLI (Seyitömer Linyit İşletmesi)    ║
║   İkinci Vaka Çalışması: Thickness + Calorific Value         ║
╠══════════════════════════════════════════════════════════════╣
║  v2 değişiklikleri:                                          ║
║  1) Variogram range kısıtı (max ~1245m = saha/3)            ║
║  2) Blok model: convex-hull içi + search radius filtresi    ║
║  3) Damar geometrisi: coal_top/coal_bot yüzeyleri           ║
║  4) Gerçekçi rezerv hacmi (3D damar geometrisi)             ║
╠══════════════════════════════════════════════════════════════╣
║  Kullanım : python sli_3dvsk_pipeline_v2.py                  ║
║  Colab    : OUTPUT_DIR değişkenini güncelleyin               ║
╚══════════════════════════════════════════════════════════════╝
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import curve_fit
from scipy.linalg import solve
from scipy.spatial import KDTree, ConvexHull, Delaunay
from scipy.interpolate import LinearNDInterpolator
from scipy import stats
import json, os, warnings
warnings.filterwarnings('ignore')

# ── Konfigürasyon ─────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────
# DATA SOURCE — IMPORTANT
# ─────────────────────────────────────────────────────────────
# The real Seyitömer-Aslanlı drill hole dataset used in the published
# manuscript is not redistributed here (see README.md, "Data Availability").
# This notebook instead uses a SYNTHETIC dataset (synthetic_seyitomer.csv,
# generated by data/synthetic_seyitomer_generator.py) that reproduces the
# same sample size, spatial extent, directional variogram structure, and
# marginal statistics reported in the manuscript (Tables 8-9). Running this
# notebook reproduces the qualitative PSD-violation / PSD-consistency
# behaviour of the three covariance formulations, but the numerical results
# will not exactly match the published tables.
#
# To reproduce the published numbers exactly, contact the corresponding
# author for access to the original dataset (see manuscript Data
# Availability Statement).
DATA_PATH      = 'synthetic_seyitomer.csv'
OUTPUT_DIR     = 'figures_sli'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DIRECTIONS     = [0, 45, 90, 135]
DIR_LABELS     = ['0° E-W','45° NE-SW','90° N-S','135° NW-SE']
ANGLE_TOL      = 22.5        # derece
LAG_WIDTH      = 200.0       # m
N_LAGS         = 8
LAG_START      = 150.0       # m
MAX_RANGE      = 1245.0      # m — saha/3, güvenilir range üst sınırı

BLOCK_SIZE     = 200.0       # m — blok boyutu
DENSITY        = 1.45        # t/m³ (Erarslan 2001, aynı saha)
CAL_CUTOFF     = 2200        # kcal/kg (santral besleme standardı, 2026)
SEARCH_RADIUS  = 600.0       # m — 3×NN ort (185m×3)
K_NN           = 20          # kriging için komşu sayısı

# ═════════════════════════════════════════════════════════════
# BÖLÜM 1 — VERİ
# ═════════════════════════════════════════════════════════════
class SLIData:
    def __init__(self, path):
        # Reads the synthetic CSV (see DATA_PATH note above). Column names
        # are mapped from the synthetic generator's schema (x, y, thickness_m,
        # calorific_kcal_kg, coal_top_masl) to this pipeline's internal
        # convention (X, Y, Z, thickness, calorific_value, coal_top).
        df = pd.read_csv(path)
        df = df.rename(columns={
            'x': 'X', 'y': 'Y',
            'thickness_m': 'thickness',
            'calorific_kcal_kg': 'calorific_value',
            'coal_top_masl': 'coal_top',
        })
        if 'Z' not in df.columns:
            df['Z'] = df['coal_top']  # drill hole collar elevation proxy
        self.df      = df
        self.n       = len(df)
        self.coords  = df[['X','Y','Z']].values
        self.thick   = df['thickness'].values
        self.cal     = df['calorific_value'].values
        self.coal_top = df['coal_top'].values
        self.coal_bot = self.coal_top - self.thick
        # Dönüşümler
        self.sqrt_thick = np.sqrt(self.thick)
        self.raw_cal    = self.cal.copy()
        self.tree       = KDTree(self.coords[:,:2])
        # Convex hull
        self.hull       = ConvexHull(self.coords[:,:2])
        self.hull_del   = Delaunay(self.coords[self.hull.vertices,:2])

    def in_hull(self, points_xy):
        """Nokta convex hull içinde mi?"""
        return self.hull_del.find_simplex(points_xy) >= 0

    def summary(self):
        nn_d,_ = self.tree.query(self.coords[:,:2], k=2)
        nn = nn_d[:,1]
        print("─"*60)
        print("VERİ ÖZETİ — SLI Seyitömer Linyit")
        print("─"*60)
        print(f"  n sondaj       : {self.n}")
        print(f"  Alan           : {self.df.X.max()-self.df.X.min():.0f}"
              f" × {self.df.Y.max()-self.df.Y.min():.0f} m")
        print(f"  Convex hull    : {self.hull.volume/1e6:.2f} km²")
        print(f"  NN mesafe      : ort={nn.mean():.1f}m "
              f"min={nn.min():.1f}m max={nn.max():.1f}m")
        print(f"  Thickness      : {self.thick.min():.1f}–"
              f"{self.thick.max():.1f}m  ort={self.thick.mean():.2f}m")
        print(f"  coal_top       : {self.coal_top.min():.1f}–"
              f"{self.coal_top.max():.1f}m")
        print(f"  coal_bot       : {self.coal_bot.min():.1f}–"
              f"{self.coal_bot.max():.1f}m")
        print(f"  Calorific      : {self.cal.min():.0f}–"
              f"{self.cal.max():.0f}  ort={self.cal.mean():.0f} kcal/kg")
        print(f"  Cut-off        : {CAL_CUTOFF} kcal/kg "
              f"(santral besleme standardı)")
        print(f"  Search radius  : {SEARCH_RADIUS:.0f}m  "
              f"Max range      : {MAX_RANGE:.0f}m")


# ═════════════════════════════════════════════════════════════
# BÖLÜM 2 — EMPİRİK VARİOGRAM
# ═════════════════════════════════════════════════════════════
class VariogramSLI:

    def __init__(self, data: SLIData):
        self.data      = data
        self.empirical = {}
        self.fitted    = {}

    @staticmethod
    def sph_gamma(h, C0, C, a):
        h = np.atleast_1d(np.array(h,dtype=float))
        return np.where(h<=0,0,
               np.where(h>=a,C0+C,
                        C0+C*(1.5*h/a-0.5*(h/a)**3)))

    def _compute(self, values, direction_deg):
        c = self.data.coords; n = self.data.n
        dir_r = np.radians(direction_deg)
        tol_r = np.radians(ANGLE_TOL)
        dx = c[:,0,None]-c[None,:,0]
        dy = c[:,1,None]-c[None,:,1]
        h_mat  = np.sqrt(dx**2+dy**2)
        pa_mat = np.arctan2(dy,dx)
        ang_ok = np.minimum(
            np.minimum(np.abs(pa_mat-dir_r),
                       np.abs(pa_mat-dir_r+np.pi)),
            np.abs(pa_mat-dir_r-np.pi)) <= tol_r
        sq  = (values[:,None]-values[None,:])**2
        tri = np.triu(np.ones((n,n),bool),k=1)
        lag_ctrs = LAG_START + np.arange(N_LAGS)*LAG_WIDTH
        h_out=[]; g_out=[]; np_out=[]
        for lh in lag_ctrs:
            lo,hi = lh-LAG_WIDTH/2, lh+LAG_WIDTH/2
            m = tri & (h_mat>=lo) & (h_mat<hi) & ang_ok
            s = sq[m]
            if len(s)>=3:
                h_out.append(lh)
                g_out.append(np.mean(s)/2.)
                np_out.append(len(s))
        return np.array(h_out),np.array(g_out),np.array(np_out)

    def _fit(self, h_data, g_data, var_total):
        """MAX_RANGE kısıtlı spherical fitting."""
        if len(h_data) < 3:
            return {"C0":var_total*0.05,"C":var_total*0.90,
                    "a":MAX_RANGE*0.7,"R2":0.0}
        try:
            popt,_ = curve_fit(
                self.sph_gamma, h_data, g_data,
                p0=[var_total*0.05, var_total*0.90,
                    min(h_data.max()*0.7, MAX_RANGE*0.6)],
                bounds=([0,1e-3,50],
                        [var_total,var_total*2,MAX_RANGE]),
                maxfev=10000)
            gp = self.sph_gamma(h_data,*popt)
            ss = np.sum((g_data-gp)**2)
            st = np.sum((g_data-g_data.mean())**2)
            r2 = 1-ss/st if st>0 else 0.
            return {"C0":popt[0],"C":popt[1],"a":popt[2],"R2":r2}
        except:
            return {"C0":var_total*0.05,"C":var_total*0.90,
                    "a":MAX_RANGE*0.6,"R2":0.0}

    def run(self):
        for vn,vals in [("thickness",self.data.sqrt_thick),
                         ("calorific",self.data.raw_cal)]:
            vt=np.var(vals); ev={}; fv={}
            for d in DIRECTIONS:
                h,g,np_=self._compute(vals,d)
                ev[d]={"h":h,"gamma":g,"n_pairs":np_}
                fv[d]=self._fit(h,g,vt)
            self.empirical[vn]=ev; self.fitted[vn]=fv
        return self

    def print_summary(self):
        print("─"*60)
        print("VARIOGRAM FİTTİNG (MAX_RANGE kısıtlı)")
        print("─"*60)
        for vn in ["thickness","calorific"]:
            tr="√" if vn=="thickness" else "raw"
            print(f"\n  {vn.upper()} (transform: {tr}):")
            print(f"  {'Yön':<15} {'C0':>10} {'C':>10} "
                  f"{'a(m)':>10} {'R²':>8}")
            print("  "+"-"*55)
            for d,lbl in zip(DIRECTIONS,DIR_LABELS):
                p=self.fitted[vn][d]
                print(f"  {lbl:<15} {p['C0']:>10.4f} "
                      f"{p['C']:>10.4f} {p['a']:>10.1f} "
                      f"{p['R2']:>8.4f}")

    def save(self, path):
        out={vn:{str(d):self.fitted[vn][d] for d in DIRECTIONS}
             for vn in ["thickness","calorific"]}
        with open(path,"w") as f: json.dump(out,f,indent=2)
        print(f"  → {path}")


# ═════════════════════════════════════════════════════════════
# BÖLÜM 3 — LMC KOVARYANS YÜZEYİ
# ═════════════════════════════════════════════════════════════
class LMCSurface:

    def __init__(self, fitted):
        self.p={}
        for vn,fv in fitted.items():
            sills=[fv[d]["C0"]+fv[d]["C"] for d in DIRECTIONS]
            nugs =[fv[d]["C0"] for d in DIRECTIONS]
            rngs ={d:fv[d]["a"] for d in DIRECTIONS}
            sn=np.mean(sills); nn=max(0.001,np.mean(nugs))
            self.p[vn]={
                "b_nug":nn,"b_str":sn-nn,
                "a_min":min(rngs.values()),
                "a_max":max(rngs.values()),
                "t_max":max(rngs,key=rngs.get),
                "sill":sn}

    def a_ell(self, az, vn):
        p=self.p[vn]; t=np.radians(az%180); tm=np.radians(p["t_max"])
        return p["a_min"]+(p["a_max"]-p["a_min"])*np.cos(t-tm)**2

    def cov(self, xi, xj, vn):
        p=self.p[vn]
        if np.allclose(xi[:2],xj[:2]): return p["b_nug"]+p["b_str"]
        dx=xj[0]-xi[0]; dy=xj[1]-xi[1]; dz=xj[2]-xi[2]
        hh=np.sqrt(dx**2+dy**2)
        az=np.degrees(np.arctan2(dy,dx))%180
        ah=self.a_ell(az,vn); av=ah*0.4
        he=(np.sqrt((hh/ah)**2+(dz/av)**2)*ah
             if ah>1e-6 else abs(dz))
        if he<=0: return p["b_str"]
        if he>=ah: return 0.
        return p["b_str"]*(1-(1.5*he/ah-0.5*(he/ah)**3))

    def diag(self,vn):
        p=self.p[vn]; return p["b_nug"]+p["b_str"]

    def psd_check(self, coords, vn):
        n=len(coords); C=np.zeros((n,n)); d=self.diag(vn)
        for i in range(n):
            C[i,i]=d
            for j in range(i+1,n):
                v=self.cov(coords[i],coords[j],vn)
                C[i,j]=v; C[j,i]=v
        eigs=np.linalg.eigvalsh(C)
        return {"lmin":eigs.min(),"cond":np.linalg.cond(C),
                "psd":eigs.min()>-1e-10}

    def print_params(self):
        print("─"*60)
        print("LMC PARAMETRELERİ")
        print("─"*60)
        for vn,p in self.p.items():
            print(f"  {vn:<12}: b_nug={p['b_nug']:.4f}  "
                  f"b_str={p['b_str']:.4f}  "
                  f"a_min={p['a_min']:.0f}m  "
                  f"a_max={p['a_max']:.0f}m  "
                  f"t_max={p['t_max']}°")


# ═════════════════════════════════════════════════════════════
# BÖLÜM 4 — KRİGİNG + LOOCV
# ═════════════════════════════════════════════════════════════
class OrdinaryKrigingSLI:

    def __init__(self, data: SLIData, surface: LMCSurface):
        self.data=data; self.surface=surface

    def predict(self, x0, idx_tr, values, vn):
        m=len(idx_tr); ct=self.data.coords[idx_tr]; vt=values[idx_tr]
        dg=self.surface.diag(vn)
        K=np.zeros((m+1,m+1))
        for i in range(m):
            K[i,i]=dg
            for j in range(i+1,m):
                v=self.surface.cov(ct[i],ct[j],vn)
                K[i,j]=v; K[j,i]=v
        K[:m,m]=1.; K[m,:m]=1.
        k0=np.array([self.surface.cov(x0,ct[i],vn)
                      for i in range(m)]+[1.])
        try: w=solve(K,k0,assume_a='sym')
        except: w=np.linalg.lstsq(K,k0,rcond=None)[0]
        return (float(np.dot(w[:m],vt)),
                max(0.,float(dg-np.dot(w,k0))))

    def loocv(self, values, vn, label=""):
        """
        LOOCV — search radius içindeki komşularla.
        """
        n=self.data.n
        preds=np.zeros(n); vars_=np.zeros(n)
        for i in range(n):
            # Search radius içindeki komşular
            idx_rad=self.data.tree.query_ball_point(
                self.data.coords[i,:2], SEARCH_RADIUS)
            idx_tr=[j for j in idx_rad if j!=i]
            if len(idx_tr)==0:  # fallback: en yakın K_NN
                _,idx_nn=self.data.tree.query(
                    self.data.coords[i,:2],k=K_NN+1)
                idx_tr=[j for j in idx_nn if j!=i][:K_NN]
            elif len(idx_tr)>K_NN*2:
                # Çok fazlaysa en yakın K_NN*2'yi al
                _,idx_nn=self.data.tree.query(
                    self.data.coords[i,:2],
                    k=min(K_NN*2+1,len(idx_rad)))
                idx_tr=[j for j in idx_nn if j!=i]
            z,s2=self.predict(self.data.coords[i],
                               idx_tr,values,vn)
            preds[i]=z; vars_[i]=s2
        res=values-preds
        r2=1-np.sum(res**2)/np.sum((values-values.mean())**2)
        msdr=float(np.mean(res**2/(vars_+1e-10)))
        return {"label":label,"pred":preds,"var":vars_,"res":res,
                "ME":float(np.mean(res)),
                "MAE":float(np.mean(np.abs(res))),
                "RMSE":float(np.sqrt(np.mean(res**2))),
                "R2":float(r2),"MSDR":msdr}


# ═════════════════════════════════════════════════════════════
# BÖLÜM 5 — BLOK MODEL (Convex-hull + damar geometrisi)
# ═════════════════════════════════════════════════════════════
class BlockModel3D:
    """
    3D Blok Modeli:
    1) Sadece convex-hull içi bloklar
    2) Sadece search radius içinde sondaj olan bloklar
    3) Damar tavan/taban yüzeylerinden gerçek 3D hacim
    """

    def __init__(self, data: SLIData, kriging: OrdinaryKrigingSLI):
        self.data=data; self.kriging=kriging
        self.blocks=None
        # Tavan ve taban yüzeyleri için interpolatörler
        xy=data.coords[:,:2]
        self.interp_top = LinearNDInterpolator(xy, data.coal_top)
        self.interp_bot = LinearNDInterpolator(xy, data.coal_bot)

    def build(self):
        df=self.data.df
        z_r=self.data.coords[:,2].mean()
        x_g=np.arange(df.X.min()+BLOCK_SIZE/2,
                       df.X.max(),BLOCK_SIZE)
        y_g=np.arange(df.Y.min()+BLOCK_SIZE/2,
                       df.Y.max(),BLOCK_SIZE)
        print(f"  Ham grid: {len(x_g)}×{len(y_g)}="
              f"{len(x_g)*len(y_g)} blok")

        bc=[]; tp=[]; cp=[]; tv=[]; cv=[]
        top_surf=[]; bot_surf=[]; thick_3d=[]
        n_hull_out=0; n_radius_out=0

        for xi in x_g:
            for yi in y_g:
                # Filtre 1: Convex hull içi
                if not self.data.in_hull([[xi,yi]]):
                    n_hull_out+=1; continue

                # Filtre 2: Search radius içinde sondaj var mı?
                idx_rad=self.data.tree.query_ball_point(
                    [xi,yi], SEARCH_RADIUS)
                if len(idx_rad)==0:
                    n_radius_out+=1; continue

                # Kriging
                x0=np.array([xi,yi,z_r])
                idx_tr=idx_rad[:K_NN*2] if len(idx_rad)>K_NN*2 else idx_rad
                t_h,t_s2=self.kriging.predict(
                    x0,idx_tr,self.data.sqrt_thick,"thickness")
                c_h,c_s2=self.kriging.predict(
                    x0,idx_tr,self.data.raw_cal,"calorific")

                # Damar yüzeyi interpolasyonu
                top_z=self.interp_top([[xi,yi]])[0]
                bot_z=self.interp_bot([[xi,yi]])[0]
                if np.isnan(top_z) or np.isnan(bot_z):
                    # Fallback: kriging thickness kullan
                    thick_z=max(0.,t_h**2)
                else:
                    thick_z=max(0.,top_z-bot_z)

                bc.append([xi,yi])
                tp.append(max(0.,t_h**2))
                cp.append(c_h)
                tv.append(t_s2)
                cv.append(c_s2)
                top_surf.append(top_z if not np.isnan(top_z)
                                  else z_r)
                bot_surf.append(bot_z if not np.isnan(bot_z)
                                  else z_r-t_h**2)
                thick_3d.append(thick_z)

        print(f"  Hull dışı elenen: {n_hull_out}")
        print(f"  Radius dışı elenen: {n_radius_out}")
        print(f"  Aktif blok: {len(bc)}")

        self.blocks={
            "coords":   np.array(bc),
            "thick_kriged": np.array(tp),    # kriging tahmini
            "thick_3d":     np.array(thick_3d), # yüzey farkı
            "cal":      np.array(cp),
            "thick_var":np.array(tv),
            "cal_var":  np.array(cv),
            "top":      np.array(top_surf),
            "bot":      np.array(bot_surf),
        }
        return self

    def reserve_summary(self):
        b=self.blocks
        # 3D hacim: yüzey interpolasyonundan thickness
        bvol_3d  = b["thick_3d"] * BLOCK_SIZE**2
        bton_3d  = bvol_3d * DENSITY
        # Kriging thickness'ten hacim (karşılaştırma için)
        bvol_k   = b["thick_kriged"] * BLOCK_SIZE**2
        bton_k   = bvol_k * DENSITY
        mask     = b["cal"] >= CAL_CUTOFF

        print("─"*60)
        print("REZERV TAHMİNİ (3D Damar Geometrisi)")
        print("─"*60)
        print(f"  Aktif blok sayısı : {len(b['coords'])}")
        print(f"  Blok boyutu       : {BLOCK_SIZE:.0f}×{BLOCK_SIZE:.0f}m")
        print(f"  Yoğunluk          : {DENSITY} t/m³")
        print(f"  Cut-off           : {CAL_CUTOFF} kcal/kg")
        print(f"\n  === TOPLAM KAYNAK (Kriging thickness) ===")
        print(f"  Alan              : {len(b['coords'])*BLOCK_SIZE**2/1e6:.2f} km²")
        print(f"  Ort. thickness    : {b['thick_kriged'].mean():.2f}m")
        print(f"  Toplam hacim      : {bvol_k.sum()/1e6:.3f} Mm³")
        print(f"  Toplam kaynak     : {bton_k.sum()/1e6:.3f} Mt")
        print(f"  Ort. calorific    : {b['cal'].mean():.0f} kcal/kg")
        print(f"\n  === TOPLAM KAYNAK (3D yüzey geometrisi) ===")
        print(f"  Ort. thickness    : {b['thick_3d'].mean():.2f}m")
        print(f"  Toplam hacim      : {bvol_3d.sum()/1e6:.3f} Mm³")
        print(f"  Toplam kaynak     : {bton_3d.sum()/1e6:.3f} Mt")
        print(f"\n  === REZERV (≥{CAL_CUTOFF} kcal/kg) ===")
        print(f"  Rezerv blok       : {mask.sum()}/{len(mask)}"
              f"  (%{mask.mean()*100:.1f})")
        print(f"  Ort. thickness    : {b['thick_kriged'][mask].mean():.2f}m")
        print(f"  Rezerv hacim (K)  : {bvol_k[mask].sum()/1e6:.3f} Mm³")
        print(f"  Rezerv (ton)      : {bton_k[mask].sum()/1e6:.3f} Mt")
        print(f"  Ort. calorific    : {b['cal'][mask].mean():.0f} kcal/kg")
        return mask, bton_k, bton_3d


# ═════════════════════════════════════════════════════════════
# BÖLÜM 6 — GÖRSELLEŞTİRME
# ═════════════════════════════════════════════════════════════
class Visualizer:

    @staticmethod
    def plot_data_hull(data: SLIData):
        """Veri + convex hull + damar geometrisi."""
        fig,axes=plt.subplots(1,3,figsize=(18,6))
        fig.patch.set_facecolor('white')
        fig.suptitle(
            "SLI Veri Özeti — Damar Geometrisi ve Convex Hull\n"
            f"n={data.n} sondaj | Kütahya, Türkiye",
            fontsize=12,fontweight='bold')

        # Thickness haritası + convex hull
        ax=axes[0]
        hull_pts=np.append(data.hull.vertices,data.hull.vertices[0])
        xy=data.coords[:,:2]
        ax.plot(xy[hull_pts,0],xy[hull_pts,1],
                'b-',lw=2,alpha=0.7,label='Convex hull')
        sc=ax.scatter(xy[:,0],xy[:,1],c=data.thick,
                       cmap='YlOrRd',s=40,edgecolors='k',
                       linewidths=0.4,zorder=4)
        plt.colorbar(sc,ax=ax,label='Thickness (m)')
        ax.set_title('Damar Kalınlığı + Convex Hull',fontsize=10)
        ax.set_xlabel('X (m)',fontsize=9)
        ax.set_ylabel('Y (m)',fontsize=9)
        ax.legend(fontsize=8); ax.grid(True,alpha=0.3)

        # Coal top
        ax=axes[1]
        sc2=ax.scatter(xy[:,0],xy[:,1],c=data.coal_top,
                        cmap='viridis',s=40,edgecolors='k',
                        linewidths=0.4)
        plt.colorbar(sc2,ax=ax,label='Tavan kotu (m)')
        ax.set_title('Damar Tavan Yüzeyi (coal_top)',fontsize=10)
        ax.set_xlabel('X (m)',fontsize=9)
        ax.set_ylabel('Y (m)',fontsize=9)
        ax.grid(True,alpha=0.3)

        # Coal bottom
        ax=axes[2]
        sc3=ax.scatter(xy[:,0],xy[:,1],c=data.coal_bot,
                        cmap='plasma',s=40,edgecolors='k',
                        linewidths=0.4)
        plt.colorbar(sc3,ax=ax,label='Taban kotu (m)')
        ax.set_title('Damar Taban Yüzeyi (coal_bot)',fontsize=10)
        ax.set_xlabel('X (m)',fontsize=9)
        ax.set_ylabel('Y (m)',fontsize=9)
        ax.grid(True,alpha=0.3)

        plt.tight_layout()
        out=os.path.join(OUTPUT_DIR,'figSLI00_data_hull.png')
        plt.savefig(out,dpi=150,bbox_inches='tight',facecolor='white')
        plt.close(); print(f"  → {out}")

    @staticmethod
    def plot_loocv(data, rt, rc, pred_bt, rmse_bt, r2_bt):
        fig,axes=plt.subplots(2,2,figsize=(14,12))
        fig.patch.set_facecolor('white')
        fig.suptitle(
            "3D-VSK — LOOCV Doğrulama: Seyitömer Linyit (SLI)\n"
            f"n={data.n} | search_radius={SEARCH_RADIUS:.0f}m | "
            f"MAX_RANGE={MAX_RANGE:.0f}m",
            fontsize=13,fontweight='bold')

        # Thickness sqrt
        ax=axes[0,0]; st=data.sqrt_thick
        lim=[st.min()-0.1,st.max()+0.1]
        sc=ax.scatter(st,rt['pred'],c=np.abs(rt['res']),
                       cmap='RdYlGn_r',vmin=0,vmax=1.5,
                       s=15,edgecolors='k',linewidths=0.3,alpha=0.8)
        ax.plot(lim,lim,'k--',lw=1.5,alpha=0.6)
        plt.colorbar(sc,ax=ax,label='|residual|')
        ax.set_xlim(lim);ax.set_ylim(lim);ax.set_aspect('equal')
        ax.set_xlabel('Gerçek √Thickness',fontsize=9)
        ax.set_ylabel('Tahmin √Thickness',fontsize=9)
        ax.set_title(
            f"Thickness (√ uzayı)\n"
            f"RMSE={rt['RMSE']:.4f}  R²={rt['R2']:.4f}  "
            f"MSDR={rt['MSDR']:.3f}",fontsize=9,fontweight='bold')
        ax.grid(True,alpha=0.3)

        # Thickness back-transform
        ax=axes[0,1]
        lim_bt=[0,data.thick.max()+2]
        sc2=ax.scatter(data.thick,pred_bt,
                        c=np.abs(data.thick-pred_bt),
                        cmap='RdYlGn_r',vmin=0,vmax=12,
                        s=15,edgecolors='k',linewidths=0.3,alpha=0.8)
        ax.plot(lim_bt,lim_bt,'k--',lw=1.5,alpha=0.6)
        plt.colorbar(sc2,ax=ax,label='|residual| (m)')
        ax.set_xlim(lim_bt);ax.set_ylim(lim_bt);ax.set_aspect('equal')
        ax.set_xlabel('Gerçek Thickness (m)',fontsize=9)
        ax.set_ylabel('Tahmin Thickness (m)',fontsize=9)
        ax.set_title(
            f"Thickness (back-transform)\n"
            f"RMSE={rmse_bt:.3f}m  R²={r2_bt:.4f}",
            fontsize=9,fontweight='bold')
        ax.grid(True,alpha=0.3)

        # Calorific
        ax=axes[1,0]
        lim_c=[data.cal.min()-100,data.cal.max()+100]
        sc3=ax.scatter(data.raw_cal,rc['pred'],
                        c=np.abs(rc['res']),cmap='RdYlGn_r',
                        vmin=0,vmax=400,s=15,edgecolors='k',
                        linewidths=0.3,alpha=0.8)
        ax.plot(lim_c,lim_c,'k--',lw=1.5,alpha=0.6)
        plt.colorbar(sc3,ax=ax,label='|residual| (kcal/kg)')
        ax.set_xlim(lim_c);ax.set_ylim(lim_c);ax.set_aspect('equal')
        ax.set_xlabel('Gerçek Calorific (kcal/kg)',fontsize=9)
        ax.set_ylabel('Tahmin Calorific (kcal/kg)',fontsize=9)
        ax.set_title(
            f"Calorific Value\n"
            f"RMSE={rc['RMSE']:.1f} kcal/kg  "
            f"R²={rc['R2']:.4f}  MSDR={rc['MSDR']:.3f}",
            fontsize=9,fontweight='bold')
        ax.grid(True,alpha=0.3)

        # Residual dağılımları
        ax=axes[1,1]
        ax.hist(rt['res'],bins=20,alpha=0.7,color='#e74c3c',
                 edgecolor='k',density=True,
                 label=f"Thickness(√) RMSE={rt['RMSE']:.3f}")
        xf=np.linspace(rt['res'].min(),rt['res'].max(),100)
        ax.plot(xf,stats.norm.pdf(
            xf,rt['res'].mean(),rt['res'].std()),'r-',lw=2)
        ax2b=ax.twinx()
        ax2b.hist(rc['res']/100,bins=20,alpha=0.5,color='#27ae60',
                   edgecolor='k',density=True,
                   label=f"Calorific/100 RMSE={rc['RMSE']:.1f}")
        ax.axvline(0,color='black',lw=2,ls='--')
        ax.set_xlabel('Residual',fontsize=9)
        ax.set_title('Residual Dağılımları',fontsize=9)
        l1,lb1=ax.get_legend_handles_labels()
        l2,lb2=ax2b.get_legend_handles_labels()
        ax.legend(l1+l2,lb1+lb2,fontsize=7.5)
        ax.grid(True,alpha=0.3)

        plt.tight_layout()
        out=os.path.join(OUTPUT_DIR,'figSLI01_loocv.png')
        plt.savefig(out,dpi=150,bbox_inches='tight',facecolor='white')
        plt.close(); print(f"  → {out}")

    @staticmethod
    def plot_block_model(data, bm, mask, bton):
        b=bm.blocks
        fig,axes=plt.subplots(1,3,figsize=(18,6))
        fig.patch.set_facecolor('white')
        hull_pts=np.append(data.hull.vertices,data.hull.vertices[0])
        xy_hull=data.coords[:,:2][hull_pts]
        fig.suptitle(
            "3D-VSK Blok Modeli — Seyitömer Linyit İşletmesi\n"
            f"{len(b['coords'])} aktif blok "
            f"({BLOCK_SIZE:.0f}×{BLOCK_SIZE:.0f}m) | "
            f"Convex-hull + search_radius={SEARCH_RADIUS:.0f}m | "
            f"Yoğunluk={DENSITY} t/m³",
            fontsize=12,fontweight='bold')

        for ax in axes:
            ax.plot(xy_hull[:,0],xy_hull[:,1],
                    'b-',lw=1.5,alpha=0.5,label='Convex hull')
            ax.scatter(data.df.X,data.df.Y,c='navy',s=5,
                        alpha=0.4,zorder=6)

        # Thickness
        ax=axes[0]
        sc=ax.scatter(b['coords'][:,0],b['coords'][:,1],
                       c=b['thick_kriged'],cmap='YlOrRd',
                       s=70,marker='s',vmin=0,vmax=35,alpha=0.85)
        plt.colorbar(sc,ax=ax,label='Thickness (m)')
        ax.set_title(
            f"Kriging: Damar Kalınlığı\n"
            f"Ort={b['thick_kriged'].mean():.1f}m  "
            f"Kaynak={bton.sum()/1e6:.1f}Mt",fontsize=10)
        ax.set_xlabel('X (m)',fontsize=9)
        ax.set_ylabel('Y (m)',fontsize=9)
        ax.legend(fontsize=7); ax.grid(True,alpha=0.2)

        # Calorific
        ax=axes[1]
        sc2=ax.scatter(b['coords'][:,0],b['coords'][:,1],
                        c=b['cal'],cmap='RdYlGn',
                        s=70,marker='s',vmin=1000,vmax=3200,alpha=0.85)
        plt.colorbar(sc2,ax=ax,label='Calorific (kcal/kg)')
        ax.set_title(
            f"Kriging: Isıl Değer\n"
            f"Ort={b['cal'].mean():.0f} kcal/kg",fontsize=10)
        ax.set_xlabel('X (m)',fontsize=9)
        ax.set_ylabel('Y (m)',fontsize=9)
        ax.grid(True,alpha=0.2)

        # Rezerv
        ax=axes[2]
        col_r=np.where(mask,b['cal'],np.nan)
        sc3=ax.scatter(b['coords'][:,0],b['coords'][:,1],
                        c=col_r,cmap='RdYlGn',
                        s=70,marker='s',vmin=1000,vmax=3200,alpha=0.85)
        n_out=(~mask).sum()
        if n_out>0:
            ax.scatter(b['coords'][~mask,0],b['coords'][~mask,1],
                        c='lightgray',s=70,marker='s',alpha=0.5,
                        label=f'<{CAL_CUTOFF} kcal/kg ({n_out})')
        plt.colorbar(sc3,ax=ax,label='Calorific (kcal/kg)')
        ax.set_title(
            f"Rezerv (≥{CAL_CUTOFF} kcal/kg)\n"
            f"{bton[mask].sum()/1e6:.2f}Mt  "
            f"Ort={b['cal'][mask].mean():.0f} kcal/kg",fontsize=10)
        ax.set_xlabel('X (m)',fontsize=9)
        ax.set_ylabel('Y (m)',fontsize=9)
        ax.legend(fontsize=7); ax.grid(True,alpha=0.2)

        plt.tight_layout()
        out=os.path.join(OUTPUT_DIR,'figSLI02_block_model.png')
        plt.savefig(out,dpi=150,bbox_inches='tight',facecolor='white')
        plt.close(); print(f"  → {out}")

    @staticmethod
    def plot_seam_surface(data, bm):
        """Damar tavan ve taban yüzeyleri."""
        b=bm.blocks
        fig,axes=plt.subplots(1,2,figsize=(14,6))
        fig.patch.set_facecolor('white')
        fig.suptitle(
            "3D-VSK — Damar Yüzeyleri (Interpolasyon)\n"
            "LinearNDInterpolator: coal_top ve coal_bot",
            fontsize=12,fontweight='bold')

        hull_pts=np.append(data.hull.vertices,data.hull.vertices[0])
        xy_hull=data.coords[:,:2][hull_pts]

        ax=axes[0]
        valid=~np.isnan(b['top'])
        sc=ax.scatter(b['coords'][valid,0],b['coords'][valid,1],
                       c=b['top'][valid],cmap='terrain',
                       s=60,marker='s',alpha=0.85)
        ax.plot(xy_hull[:,0],xy_hull[:,1],'b-',lw=1.5,alpha=0.5)
        ax.scatter(data.df.X,data.df.Y,c='red',s=8,zorder=6,
                    alpha=0.6,label='Sondaj')
        plt.colorbar(sc,ax=ax,label='Tavan kotu (m)')
        ax.set_title('Damar Tavan Yüzeyi (Kriged)',fontsize=10)
        ax.set_xlabel('X (m)',fontsize=9)
        ax.set_ylabel('Y (m)',fontsize=9)
        ax.legend(fontsize=8); ax.grid(True,alpha=0.3)

        ax=axes[1]
        valid2=~np.isnan(b['bot'])
        sc2=ax.scatter(b['coords'][valid2,0],b['coords'][valid2,1],
                        c=b['bot'][valid2],cmap='terrain',
                        s=60,marker='s',alpha=0.85)
        ax.plot(xy_hull[:,0],xy_hull[:,1],'b-',lw=1.5,alpha=0.5)
        ax.scatter(data.df.X,data.df.Y,c='red',s=8,zorder=6,alpha=0.6)
        plt.colorbar(sc2,ax=ax,label='Taban kotu (m)')
        ax.set_title('Damar Taban Yüzeyi (Kriged)',fontsize=10)
        ax.set_xlabel('X (m)',fontsize=9)
        ax.set_ylabel('Y (m)',fontsize=9)
        ax.grid(True,alpha=0.3)

        plt.tight_layout()
        out=os.path.join(OUTPUT_DIR,'figSLI03_seam_surface.png')
        plt.savefig(out,dpi=150,bbox_inches='tight',facecolor='white')
        plt.close(); print(f"  → {out}")


# ═════════════════════════════════════════════════════════════
# ANA PIPELINE
# ═════════════════════════════════════════════════════════════
def run_pipeline():
    print("\n"+"═"*62)
    print("  3D-VSK SLI PIPELINE v2 — Başlatılıyor")
    print("═"*62)

    # Faz 0
    print("\n[FAZ 0] Veri yükleniyor...")
    data=SLIData(DATA_PATH); data.summary()
    print("\n  Görsel oluşturuluyor...")
    Visualizer.plot_data_hull(data)

    # Faz 1
    print("\n[FAZ 1] Empirik variogram (MAX_RANGE kısıtlı)...")
    vario=VariogramSLI(data); vario.run()
    vario.print_summary()
    vario.save(os.path.join(OUTPUT_DIR,'sli_variogram_params.json'))

    # Faz 2
    print("\n[FAZ 2] LMC yüzeyi...")
    surf=LMCSurface(vario.fitted); surf.print_params()

    # Faz 3
    print("\n[FAZ 3] PSD Analizi (örneklem n=40)...")
    print(f"  {'Değişken':<15} {'λ_min':>10} {'Koşul':>10} {'PSD':>6}")
    print("  "+"-"*44)
    np.random.seed(42)
    idx_s=np.random.choice(data.n,min(40,data.n),replace=False)
    for vn in ["thickness","calorific"]:
        r=surf.psd_check(data.coords[idx_s],vn)
        sym="✓" if r["psd"] else "✗"
        print(f"  {vn:<15} {r['lmin']:>10.5f} {r['cond']:>10.2f} {sym:>6}")

    # Faz 4
    print(f"\n[FAZ 4] LOOCV (n={data.n}, "
          f"search_r={SEARCH_RADIUS:.0f}m)...")
    ok=OrdinaryKrigingSLI(data,surf)
    loocv_t=ok.loocv(data.sqrt_thick,"thickness","Thickness(√)")
    loocv_c=ok.loocv(data.raw_cal,"calorific","Calorific")
    pred_bt=loocv_t["pred"]**2
    rmse_bt=np.sqrt(np.mean((data.thick-pred_bt)**2))
    r2_bt=(1-np.sum((data.thick-pred_bt)**2)/
            np.sum((data.thick-data.thick.mean())**2))

    print(f"\n  {'Değişken':<22} {'ME':>8} {'RMSE':>8} "
          f"{'R²':>8} {'MSDR':>8}")
    print("  "+"-"*52)
    for r in [loocv_t,loocv_c]:
        print(f"  {r['label']:<22} {r['ME']:>+8.4f} "
              f"{r['RMSE']:>8.4f} {r['R2']:>8.4f} "
              f"{r['MSDR']:>8.4f}")
    print(f"\n  Thickness (m): RMSE={rmse_bt:.3f}m  R²={r2_bt:.4f}")
    print("\n  Görsel oluşturuluyor...")
    Visualizer.plot_loocv(data,loocv_t,loocv_c,pred_bt,rmse_bt,r2_bt)

    # Faz 5
    print("\n[FAZ 5] 3D Blok modeli inşa ediliyor...")
    bm=BlockModel3D(data,ok); bm.build()
    mask,bton,bton_3d=bm.reserve_summary()
    print("\n  Görseller oluşturuluyor...")
    Visualizer.plot_block_model(data,bm,mask,bton)
    Visualizer.plot_seam_surface(data,bm)

    # Özet
    print("\n"+"═"*62)
    print("  SLI 3D-VSK PIPELINE v2 — Tamamlandı")
    print("═"*62)
    print(f"\n  LOOCV özeti:")
    print(f"    Thickness (√) : RMSE={loocv_t['RMSE']:.4f}  "
          f"R²={loocv_t['R2']:.4f}")
    print(f"    Thickness (m) : RMSE={rmse_bt:.3f}m  R²={r2_bt:.4f}")
    print(f"    Calorific     : RMSE={loocv_c['RMSE']:.1f}  "
          f"R²={loocv_c['R2']:.4f}")
    print(f"\n  Çıktılar: {OUTPUT_DIR}/")


if __name__ == "__main__":
    run_pipeline()


## Cell 4 — Variogram surface visualisation
3D/contour variogram surface plots and azimuthal sensitivity analysis (Kalgoorlie dataset).


In [ ]:
"""
3D-VSK — Yüzey Görselleştirme + Açı Analizi
============================================
Pipeline'a entegre edilecek modül.
İki bağımsız çalıştırılabilir bölüm:

  A) plot_variogram_surface()   → fig05, fig06, fig07
  B) plot_angle_sensitivity()   → fig08 (hakem sorusuna cevap)

Tekrarlanabilirlik: tüm parametreler Faz 0 fitted değerlerden gelir.
Bağımlılık: numpy, matplotlib (scipy gerekmez)
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import os, warnings
warnings.filterwarnings('ignore')

# ── Konfigürasyon ─────────────────────────────────────────────
OUTPUT_DIR = "figures"   # Colab: "/content/drive/MyDrive/3dvsk/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Faz 0 fitted parametreler (variogram_params.json'dan) ─────
VP = {
      0: {"C0":0.00000, "C":0.16752, "a":28.111},
     45: {"C0":0.02828, "C":0.29606, "a":62.220},
     90: {"C0":0.03054, "C":0.29606, "a":66.016},
    135: {"C0":0.00000, "C":0.06440, "a":48.307},
}
DIRS = sorted(VP.keys())

# LMC parametreleri (Faz 3'ten)
B_NUG    = 0.01470
B_STR    = 0.20601
A_MIN    = 28.111
A_MAX    = 66.016
THETA_MAX = 90.0   # en uzun range yönü (°)

# ═════════════════════════════════════════════════════════════
# YARDIMCI FONKSİYONLAR
# ═════════════════════════════════════════════════════════════

def sph_gamma(h, C0, C, a):
    """Spherical variogram γ(h)."""
    if h <= 0:  return 0.0
    if h >= a:  return C0 + C
    return C0 + C*(1.5*h/a - 0.5*(h/a)**3)

def nearest_dir(ang_deg):
    """En yakın variogram yönünü seç (ayrıksal)."""
    base  = ang_deg % 180
    diffs = [min(abs(base-d), 180-abs(base-d)) for d in DIRS]
    return DIRS[np.argmin(diffs)]

def a_ellipse(az_deg):
    """Elipsoidal anizotropi: yöne bağlı range."""
    t  = np.radians(az_deg % 180)
    tm = np.radians(THETA_MAX)
    return A_MIN + (A_MAX - A_MIN) * np.cos(t - tm)**2

def gamma_discrete(h, az):
    """Ayrıksal model seçimi — γ(h, az)."""
    d = nearest_dir(az)
    p = VP[d]
    return sph_gamma(h, p["C0"], p["C"], p["a"])

def gamma_bilinear(h, az, dirs_used=None, vp_used=None):
    """
    Bilinear yüzey interpolasyonu — γ(h, az).
    dirs_used: kullanılan yön listesi (varsayılan: DIRS)
    """
    if dirs_used is None: dirs_used = DIRS
    if vp_used   is None: vp_used   = VP
    base  = az % 180
    de    = dirs_used + [180]
    vpe   = dict(vp_used); vpe[180] = vp_used[dirs_used[0]]
    lo, hi = de[-2], de[-1]
    for k in range(len(de)-1):
        if de[k] <= base <= de[k+1]:
            lo, hi = de[k], de[k+1]; break
    p_lo = vpe[lo]; p_hi = vpe[hi]
    t    = (base-lo)/(hi-lo) if hi != lo else 0.0
    g_lo = sph_gamma(h, p_lo["C0"], p_lo["C"], p_lo["a"])
    g_hi = sph_gamma(h, p_hi["C0"], p_hi["C"], p_hi["a"])
    return (1-t)*g_lo + t*g_hi

def gamma_lmc(h, az):
    """LMC elipsoidal — γ(h, az)."""
    a_h = a_ellipse(az)
    if h <= 0:    return 0.0
    if h >= a_h:  return B_STR
    return B_STR*(1.5*h/a_h - 0.5*(h/a_h)**3)

# ═════════════════════════════════════════════════════════════
# A) VARIOGRAM YÜZEYİ GÖRSELLEŞTİRME
# ═════════════════════════════════════════════════════════════

def plot_variogram_surface():
    """
    3 yöntemin variogram yüzeyini karşılaştırmalı göster.
    Erarslan (2001) Fig 3 ile doğrudan karşılaştırılabilir.

    Üretilen dosyalar:
        fig05_variogram_surface_3d.png
        fig06_variogram_surface_contour.png
        fig07_surface_difference.png
    """
    h_arr   = np.linspace(0, 68, 60)
    ang_arr = np.linspace(0, 180, 72)
    H, A    = np.meshgrid(h_arr, ang_arr)

    # Yüzeyleri hesapla
    Z = {}
    Z["disc"] = np.array([[gamma_discrete(h,a)  for h in h_arr] for a in ang_arr])
    Z["bil"]  = np.array([[gamma_bilinear(h,a)  for h in h_arr] for a in ang_arr])
    Z["lmc"]  = np.array([[gamma_lmc(h,a)       for h in h_arr] for a in ang_arr])

    methods = [
        ("disc", "Classical Discrete\n(ayrıksal seçim — baseline)",
         "Süreksiz sıçramalar\nPSD: λ_min=−0.265"),
        ("bil",  "Classical Bilinear\n(Erarslan 2001)",
         "Türev süreksizliği\nPSD: λ_min=−0.124"),
        ("lmc",  "3D-VSK LMC-Ellipsoidal\n(Bu çalışma)",
         "Tam smooth + PSD garantili\nλ_min=+0.019"),
    ]

    # ── Fig 5: 3D yüzey ──────────────────────────────────────
    fig = plt.figure(figsize=(18, 6))
    fig.patch.set_facecolor('white')
    fig.suptitle(
        "3D Variogram Yüzeyi: γ(h, θ)\n"
        "Lag mesafesi (h) × Yön (θ) → Variogram değeri",
        fontsize=13, fontweight='bold')

    for idx, (key, title, note) in enumerate(methods):
        ax = fig.add_subplot(1, 3, idx+1, projection='3d')
        surf = ax.plot_surface(H, A, Z[key],
                                cmap='YlOrRd', alpha=0.85,
                                linewidth=0, antialiased=True)
        # Fitted variogram modellerini üzerine çiz (lacivert)
        for d in DIRS:
            p = VP[d]
            g_line = [sph_gamma(h, p["C0"], p["C"], p["a"]) for h in h_arr]
            ax.plot(h_arr, [d]*len(h_arr), g_line,
                    '-', color='navy', lw=1.8, alpha=0.8,
                    zorder=5)

        ax.set_xlabel('h (m)', fontsize=8, labelpad=6)
        ax.set_ylabel('θ (°)', fontsize=8, labelpad=6)
        ax.set_zlabel('γ(h,θ)', fontsize=8, labelpad=6)
        ax.set_yticks([0,45,90,135,180])
        ax.set_title(title, fontsize=9, fontweight='bold', pad=8)
        ax.text2D(0.04, 0.03, note, transform=ax.transAxes,
                   fontsize=7.5, color='#2c3e50',
                   bbox=dict(boxstyle='round,pad=0.3',
                              fc='white', alpha=0.85))
        fig.colorbar(surf, ax=ax, shrink=0.45, label='γ')
        ax.view_init(elev=28, azim=-50)

    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, 'fig05_variogram_surface_3d.png')
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"  → {out}")

    # ── Fig 6: Kontur + yön kesitleri ────────────────────────
    fig2, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig2.patch.set_facecolor('white')
    fig2.suptitle(
        "3D Variogram Yüzeyi: Kontur ve Yön Kesitleri\n"
        "Kalgoorlie Au — Fitted parametreler (Faz 0)",
        fontsize=12, fontweight='bold')

    for idx, (key, title, _) in enumerate(methods):
        # Kontur
        ax = axes[0, idx]
        cf = ax.contourf(H, A, Z[key], levels=20,
                          cmap='YlOrRd', vmin=0, vmax=0.35)
        for d in DIRS:
            ax.axhline(d, color='navy', lw=1.5, ls='--', alpha=0.6)
        ax.set_yticks([0,45,90,135,180])
        ax.set_xlabel('Lag h (m)', fontsize=9)
        ax.set_ylabel('Yön θ (°)', fontsize=9)
        ax.set_title(f'Kontur: {title.split(chr(10))[0]}',
                      fontsize=9, fontweight='bold')
        plt.colorbar(cf, ax=ax, label='γ(h,θ)')
        ax.grid(True, alpha=0.2)

        # Yön kesitleri
        ax2 = axes[1, idx]
        ang_fine  = np.linspace(0, 180, 360)
        h_cuts    = [10, 25, 45]
        hcut_cols = ['#3498db','#e74c3c','#2ecc71']
        gamma_func = {"disc": gamma_discrete,
                       "bil":  gamma_bilinear,
                       "lmc":  gamma_lmc}[key]
        for h_f, hcol in zip(h_cuts, hcut_cols):
            vals = [gamma_func(h_f, a) for a in ang_fine]
            ax2.plot(ang_fine, vals, '-', color=hcol,
                      lw=2.2, label=f'h={h_f}m')
        for d in DIRS:
            ax2.axvline(d, color='gray', ls=':', lw=1, alpha=0.5)
            ax2.text(d, 0.355, f'{d}°', fontsize=7.5,
                      ha='center', color='gray')
        ax2.set_xlabel('Yön θ (°)', fontsize=9)
        ax2.set_ylabel('γ(h, θ)', fontsize=9)
        ax2.set_xticks([0,45,90,135,180])
        ax2.set_title(f'Yön Kesiti: {title.split(chr(10))[0]}',
                       fontsize=9)
        ax2.legend(fontsize=8)
        ax2.grid(True, alpha=0.3)
        ax2.set_xlim(0, 180); ax2.set_ylim(0, 0.38)

    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, 'fig06_variogram_surface_contour.png')
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"  → {out}")

    # ── Fig 7: Fark yüzeyleri ────────────────────────────────
    fig3, axes3 = plt.subplots(1, 3, figsize=(16, 5))
    fig3.patch.set_facecolor('white')
    fig3.suptitle(
        "Variogram Yüzeyi Fark Analizi\n"
        "3D-VSK (LMC) ile Klasik Yaklaşımlar Arasındaki Δγ",
        fontsize=12, fontweight='bold')

    diffs = [
        (Z["disc"]-Z["lmc"], "Discrete − LMC", '#e74c3c'),
        (Z["bil"] -Z["lmc"], "Bilinear − LMC", '#e67e22'),
        (Z["bil"] -Z["disc"],"Bilinear − Discrete\n(süreksizlik haritası)",'#3498db'),
    ]
    for idx, (Zdiff, title, col) in enumerate(diffs):
        ax = axes3[idx]
        vmax = max(np.abs(Zdiff).max(), 0.01)
        cf = ax.contourf(H, A, Zdiff, levels=20,
                          cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        ax.contour(H, A, Zdiff, levels=[0], colors='k',
                    linewidths=1.5, linestyles='--')
        plt.colorbar(cf, ax=ax, label='Δγ')
        ax.set_yticks([0,45,90,135,180])
        ax.set_xlabel('Lag h (m)', fontsize=9)
        ax.set_ylabel('Yön θ (°)', fontsize=9)
        ax.set_title(title, fontsize=9, fontweight='bold')
        ax.grid(True, alpha=0.2)
        imax,jmax = np.unravel_index(np.abs(Zdiff).argmax(),Zdiff.shape)
        ax.scatter(H[imax,jmax], A[imax,jmax], s=100,
                    color='gold', edgecolors='k', zorder=6,
                    label=f'Max |Δ|={abs(Zdiff[imax,jmax]):.4f}')
        ax.legend(fontsize=8)

    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, 'fig07_surface_difference.png')
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"  → {out}")


# ═════════════════════════════════════════════════════════════
# B) AÇI DUYARLILIĞI ANALİZİ (hakem sorusuna cevap)
# ═════════════════════════════════════════════════════════════

def plot_angle_sensitivity():
    """
    Soru: Daha sık açı aralığı (15°, 22.5°, 45°) yüzeyi
    iyileştirir mi?

    Analiz:
      - Farklı açı adımlarıyla bilinear yüzey oluştur
      - LMC yüzeyiyle karşılaştır (referans)
      - n_pairs kısıtını göster

    Üretilen dosya: fig08_angle_sensitivity.png
    """
    h_arr    = np.linspace(0, 68, 60)
    ang_arr  = np.linspace(0, 180, 180)  # 1° çözünürlük
    H2, A2   = np.meshgrid(h_arr, ang_arr)

    # ── Farklı açı adımı senaryoları ─────────────────────────
    # Senaryo A: 45° adım (Erarslan — 4 yön)
    dirs_45  = [0, 45, 90, 135]

    # Senaryo B: 22.5° adım (8 yön — interpolasyon)
    # Gerçek ölçüm yok, ara yönleri bilinear ile tahmin et
    dirs_225 = [0, 22, 45, 67, 90, 112, 135, 157]

    # Senaryo C: 15° adım (12 yön — interpolasyon)
    dirs_15  = list(range(0, 180, 15))

    # Senaryo D: LMC (referans — sürekli)
    # Her senaryo için VP (22.5° ve 15° için interpolasyon)
    def make_vp_interpolated(dirs_new, vp_ref, dirs_ref):
        """
        Mevcut 4 yönden yeni yönler için VP interpolasyonu.
        Gerçek çok yönlü ölçüm olmadığında bu tahmini değerler.
        """
        vp_new = {}
        for d in dirs_new:
            base = d % 180
            # En yakın iki referans yön arasında bilinear
            d_ext = dirs_ref + [180]
            vp_ext = dict(vp_ref); vp_ext[180] = vp_ref[dirs_ref[0]]
            lo, hi = d_ext[-2], d_ext[-1]
            for k in range(len(d_ext)-1):
                if d_ext[k] <= base <= d_ext[k+1]:
                    lo, hi = d_ext[k], d_ext[k+1]; break
            p_lo = vp_ext[lo]; p_hi = vp_ext[hi]
            t = (base-lo)/(hi-lo) if hi != lo else 0.0
            C0_i = (1-t)*p_lo["C0"] + t*p_hi["C0"]
            C_i  = (1-t)*p_lo["C"]  + t*p_hi["C"]
            a_i  = (1-t)*p_lo["a"]  + t*p_hi["a"]
            vp_new[d] = {"C0":C0_i, "C":C_i, "a":a_i}
        return vp_new

    vp_225 = make_vp_interpolated(dirs_225, VP, DIRS)
    vp_15  = make_vp_interpolated(dirs_15,  VP, DIRS)

    # Yüzeyler
    Z_45  = np.array([[gamma_bilinear(h,a,dirs_45,VP)     for h in h_arr]
                        for a in ang_arr])
    Z_225 = np.array([[gamma_bilinear(h,a,dirs_225,vp_225) for h in h_arr]
                        for a in ang_arr])
    Z_15  = np.array([[gamma_bilinear(h,a,dirs_15,vp_15)   for h in h_arr]
                        for a in ang_arr])
    Z_lmc = np.array([[gamma_lmc(h,a) for h in h_arr] for a in ang_arr])

    # ── n_pairs analizi ───────────────────────────────────────
    # Kalgoorlie verisinde gerçek çift sayıları
    coords_xy = np.array([
        [4120.40,8110.20],[4130.10,8141.40],[4140.80,8128.80],
        [4151.00,8133.10],[4102.10,8109.80],[4141.20,8157.10],
        [4130.90,8136.10],[4151.00,8110.60],[4101.50,8145.80],
        [4111.30,8130.90],[4122.10,8120.40],[4113.40,8151.20],
        [4142.80,8133.30],[4153.10,8151.60],[4153.60,8152.10],
        [4142.90,8122.70],[4123.40,8130.30],[4133.80,8151.80],
        [4103.20,8140.10],[4114.50,8133.40],
    ])
    n = len(coords_xy)

    def count_pairs(angle_tol_deg, lag_width=8.0, n_lags=7, lag_start=6.0):
        """Verilen toleransta her lag için çift sayısı."""
        tol_r   = np.radians(angle_tol_deg)
        results = {}
        for dir_deg in [0, 45, 90, 135]:
            dir_r  = np.radians(dir_deg)
            lags   = lag_start + np.arange(n_lags)*lag_width
            npairs = []
            for lag_h in lags:
                lo, hi = lag_h - lag_width/2, lag_h + lag_width/2
                cnt = 0
                for i in range(n):
                    for j in range(i+1, n):
                        dx = coords_xy[j,0]-coords_xy[i,0]
                        dy = coords_xy[j,1]-coords_xy[i,1]
                        h  = np.sqrt(dx**2+dy**2)
                        if lo <= h < hi:
                            pair_ang = np.arctan2(dy,dx)
                            diffs = [abs(pair_ang-dir_r),
                                     abs(pair_ang-dir_r+np.pi),
                                     abs(pair_ang-dir_r-np.pi)]
                            if min(diffs) <= tol_r:
                                cnt += 1
                npairs.append(cnt)
            results[dir_deg] = npairs
        return results, lags

    pairs_45,  lags = count_pairs(22.5)
    pairs_225, _    = count_pairs(11.25)
    pairs_15,  _    = count_pairs(7.5)

    # ── Görselleştirme ────────────────────────────────────────
    fig4 = plt.figure(figsize=(18, 12))
    fig4.patch.set_facecolor('white')
    fig4.suptitle(
        "Açı Aralığı Duyarlılık Analizi\n"
        "Hakem Sorusu: 'Daha sık yön örneklemesi yüzeyi iyileştirir mi?'",
        fontsize=13, fontweight='bold')

    gs = plt.GridSpec(2, 4, figure=fig4, hspace=0.48, wspace=0.38)

    # Satır 1: Yüzey karşılaştırması (kontur)
    scenarios = [
        (Z_45,  "45° adım — 4 yön\n(Erarslan 2001)", '#e74c3c'),
        (Z_225, "22.5° adım — 8 yön\n(interpolasyon)", '#e67e22'),
        (Z_15,  "15° adım — 12 yön\n(interpolasyon)", '#3498db'),
        (Z_lmc, "LMC Sürekli\n(referans — bu çalışma)", '#27ae60'),
    ]
    for idx, (Zs, title, col) in enumerate(scenarios):
        ax = fig4.add_subplot(gs[0, idx])
        cf = ax.contourf(H2, A2, Zs, levels=20,
                          cmap='YlOrRd', vmin=0, vmax=0.35)
        plt.colorbar(cf, ax=ax, label='γ(h,θ)')
        ax.set_yticks([0,45,90,135,180])
        ax.set_xlabel('Lag h (m)', fontsize=9)
        ax.set_ylabel('Yön θ (°)', fontsize=9)
        ax.set_title(title, fontsize=9, fontweight='bold')
        ax.grid(True, alpha=0.2)

    # Satır 2 sol-orta: h=25m yön kesiti karşılaştırması
    ax_cut = fig4.add_subplot(gs[1, :2])
    ang_fine = np.linspace(0, 180, 360)
    h_cut = 25.0
    cut_data = [
        ([gamma_bilinear(h_cut,a,dirs_45,VP)     for a in ang_fine],
         '45° / 4 yön (Erarslan)', '#e74c3c', '--', 2.0),
        ([gamma_bilinear(h_cut,a,dirs_225,vp_225) for a in ang_fine],
         '22.5° / 8 yön (interpolasyon)', '#e67e22', '-.', 2.0),
        ([gamma_bilinear(h_cut,a,dirs_15,vp_15)   for a in ang_fine],
         '15° / 12 yön (interpolasyon)', '#3498db', ':', 2.0),
        ([gamma_lmc(h_cut,a) for a in ang_fine],
         'LMC Sürekli (referans)', '#27ae60', '-', 2.5),
    ]
    for vals, lbl, col, ls, lw in cut_data:
        ax_cut.plot(ang_fine, vals, ls=ls, color=col, lw=lw, label=lbl)
    for d in DIRS:
        ax_cut.axvline(d, color='gray', ls=':', lw=0.8, alpha=0.5)
    ax_cut.set_xlabel('Yön θ (°)', fontsize=10)
    ax_cut.set_ylabel(f'γ(h={h_cut}m, θ)', fontsize=10)
    ax_cut.set_title(f'Yön Kesiti h={h_cut}m — Açı Adımı Karşılaştırması',
                      fontsize=10)
    ax_cut.set_xticks([0,45,90,135,180])
    ax_cut.legend(fontsize=8.5); ax_cut.grid(True, alpha=0.3)
    ax_cut.set_xlim(0,180)

    # Satır 2 sağ: n_pairs tablosu
    ax_np = fig4.add_subplot(gs[1, 2:])
    ax_np.axis('off')

    # Tablo verisi
    lag_labels = [f'{int(l)}m' for l in lags]
    tol_scenarios = [
        ("45° adım\n(tol=22.5°)", pairs_45),
        ("22.5° adım\n(tol=11.25°)", pairs_225),
        ("15° adım\n(tol=7.5°)", pairs_15),
    ]
    dir_labels_short = ['0°','45°','90°','135°']

    # Her senaryo için ortalama çift sayısı
    rows_tbl = []
    for scen_name, pairs_dict in tol_scenarios:
        row = [scen_name.replace('\n',' ')]
        all_vals = []
        for d in [0,45,90,135]:
            vals = pairs_dict[d]
            avg = np.mean(vals)
            min_val = min(vals)
            all_vals.extend(vals)
            row.append(f"avg={avg:.1f}\nmin={min_val}")
        row.append(f"Toplam ort: {np.mean(all_vals):.1f}")
        rows_tbl.append(row)

    t = ax_np.table(
        cellText=rows_tbl,
        colLabels=['Senaryo','0°','45°','90°','135°','Genel Ort.'],
        cellLoc='center', loc='center')
    t.auto_set_font_size(False); t.set_fontsize(8.5); t.scale(1.1, 2.2)
    for j in range(6):
        t[0,j].set_facecolor('#2c3e50')
        t[0,j].set_text_props(color='white', fontweight='bold')
    # Düşük çift sayılarını kırmızı işaretle
    for i,(_,pairs_dict) in enumerate(tol_scenarios):
        for jj,d in enumerate([0,45,90,135]):
            if min(pairs_dict[d]) < 3:
                t[i+1,jj+1].set_facecolor('#fadbd8')
                t[i+1,jj+1].set_text_props(color='#c0392b')

    ax_np.set_title(
        'n_pairs Analizi: Açı Toleransına Göre Çift Sayısı\n'
        '(Kırmızı: min < 3 — variogram güvenilmez)',
        fontsize=10, fontweight='bold', pad=10)

    # ── Sonuç kutusu ─────────────────────────────────────────
    fig4.text(0.5, 0.01,
        "SONUÇ: Daha sık açı örneklemesi teorik üstünlük sağlar. "
        "Ancak n=20 veri setinde tol<11° ile çift sayısı kritik "
        "sınırın altına düşer (min<3). 3D-VSK/LMC bu kısıtı "
        "aşar: mevcut yön sayısından bağımsız olarak sürekli "
        "kovaryans sağlar.",
        ha='center', fontsize=9.5, style='italic',
        bbox=dict(boxstyle='round,pad=0.5', fc='#eaf4fb', ec='#2980b9'))

    out = os.path.join(OUTPUT_DIR, 'fig08_angle_sensitivity.png')
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"  → {out}")


# ═════════════════════════════════════════════════════════════
# ANA ÇALIŞMA
# ═════════════════════════════════════════════════════════════
if __name__ == "__main__":
    print("\n" + "═"*55)
    print("  Yüzey Görselleştirme + Açı Analizi")
    print("═"*55)

    print("\n[A] Variogram yüzeyleri...")
    plot_variogram_surface()

    print("\n[B] Açı duyarlılık analizi...")
    plot_angle_sensitivity()

    print("\n  Tüm görseller tamamlandı.")